# 06 — Recalcular os resultados históricos usados no pôster

Pode ser executado sem GPU e sem imagens. Mantém os resultados anteriores separados dos novos experimentos UTKFace.


In [ ]:
%pip -q install scikit-learn pandas matplotlib scipy


In [ ]:
import numpy as np,pandas as pd
from sklearn.metrics import roc_auc_score
from math import sqrt

def wilson(k,n):
    if not n:return (np.nan,np.nan)
    z=1.95996398454;p=k/n;d=1+z*z/n
    center=(p+z*z/(2*n))/d;half=z*sqrt(p*(1-p)/n+z*z/(4*n*n))/d
    return center-half,center+half

def summarize(d):
    y=d.label.to_numpy();p=d.pred.to_numpy()
    tn=int(((y==0)&(p==0)).sum());fp=int(((y==0)&(p==1)).sum())
    fn=int(((y==1)&(p==0)).sum());tp=int(((y==1)&(p==1)).sum())
    div=lambda a,b:a/b if b else np.nan
    tpr=div(tp,tp+fn);tnr=div(tn,tn+fp)
    return dict(n=len(d),n_real=tn+fp,n_fake=tp+fn,tn=tn,fp=fp,fn=fn,tp=tp,
        accuracy=div(tp+tn,len(d)),balanced_accuracy=(tpr+tnr)/2,
        precision=div(tp,tp+fp),recall=tpr,f1=div(2*tp,2*tp+fp+fn),
        fpr=div(fp,fp+tn),fnr=div(fn,fn+tp),
        auc=roc_auc_score(y,d.p_fake) if len(set(y))==2 and 'p_fake' in d else np.nan)

def table_metrics(df):
    rows=[]
    for condition,data in df.groupby('condition'):
        for axis in ['all','gender','race','intersection']:
            groups=[('Todos',data)] if axis=='all' else data.groupby(axis,dropna=False)
            for group,d in groups:
                row=summarize(d);row.update(condition=condition,axis=axis,group=group)
                eligible=d[d.clean_pred==d.label] if 'clean_pred' in d else d.iloc[:0]
                row.update(asr=(eligible.pred!=eligible.label).mean() if len(eligible) else np.nan,
                           n_asr=len(eligible))
                rows.append(row)
    return pd.DataFrame(rows)


## 1. Dados recebidos
Os três CSVs abaixo são cópias integrais dos arquivos em `tcc_si.zip`. Não reexecutam o modelo. Os 307 nomes serão conferidos contra o split de teste. Não existe coluna de raça nesses resultados: não é possível criar retrospectivamente resultados raciais.


In [ ]:
import io
clean=pd.read_csv(io.StringIO('filename,label,gender,pred,conf,acerto\nreal_00031.jpg,0,homem,0,0.594,1\nreal_00037.jpg,0,homem,1,0.5462,0\nreal_00038.jpg,0,homem,1,0.7542,0\nreal_00060.jpg,0,homem,0,0.8344,1\nreal_00072.jpg,0,homem,1,0.5983,0\nreal_00087.jpg,0,homem,0,0.5529,1\nreal_00090.jpg,0,homem,0,0.5409,1\nreal_00093.jpg,0,homem,0,0.5861,1\nreal_00094.jpg,0,homem,0,0.5058,1\nreal_00099.jpg,0,homem,0,0.9462,1\nreal_00105.jpg,0,homem,0,0.7679,1\nreal_00159.jpg,0,homem,0,0.5567,1\nreal_00162.jpg,0,homem,0,0.5674,1\nreal_00178.jpg,0,homem,1,0.5459,0\nreal_00179.jpg,0,homem,0,0.6279,1\nreal_00227.jpg,0,homem,0,0.8569,1\nreal_00237.jpg,0,homem,0,0.6019,1\nreal_00254.jpg,0,homem,0,0.5807,1\nreal_00259.jpg,0,homem,0,0.7274,1\nreal_00264.jpg,0,homem,1,0.6782,0\nreal_00390.jpg,0,homem,0,0.8234,1\nreal_00392.jpg,0,homem,1,0.5551,0\nreal_00400.jpg,0,homem,0,0.7126,1\nreal_00406.jpg,0,homem,0,0.7204,1\nreal_00412.jpg,0,homem,0,0.8546,1\nreal_00418.jpg,0,homem,0,0.7092,1\nreal_00453.jpg,0,homem,0,0.5273,1\nreal_00459.jpg,0,homem,0,0.7855,1\nreal_00462.jpg,0,homem,0,0.8317,1\nreal_00492.jpg,0,homem,0,0.83,1\nreal_00500.jpg,0,homem,0,0.8438,1\nreal_00506.jpg,0,homem,0,0.843,1\nreal_00530.jpg,0,homem,0,0.7483,1\nreal_00532.jpg,0,homem,0,0.6882,1\nreal_00534.jpg,0,homem,1,0.6268,0\nreal_00535.jpg,0,homem,1,0.8855,0\nreal_00556.jpg,0,homem,0,0.5868,1\nreal_00566.jpg,0,homem,0,0.8142,1\nreal_00571.jpg,0,homem,1,0.7831,0\nreal_00582.jpg,0,homem,0,0.6571,1\nreal_00590.jpg,0,homem,0,0.6732,1\nreal_00594.jpg,0,homem,1,0.5571,0\nreal_00598.jpg,0,homem,0,0.7881,1\nreal_00604.jpg,0,homem,0,0.6043,1\nreal_00605.jpg,0,homem,0,0.7259,1\nreal_00610.jpg,0,homem,0,0.7961,1\nreal_00623.jpg,0,homem,0,0.737,1\nreal_00654.jpg,0,homem,0,0.7549,1\nreal_00666.jpg,0,homem,0,0.7031,1\nreal_00674.jpg,0,homem,0,0.5341,1\nreal_00686.jpg,0,homem,0,0.8071,1\nreal_00698.jpg,0,homem,0,0.6597,1\nreal_00702.jpg,0,homem,0,0.6199,1\nreal_00706.jpg,0,homem,0,0.8875,1\nreal_00722.jpg,0,homem,0,0.8288,1\nreal_00741.jpg,0,homem,1,0.5743,0\nreal_00756.jpg,0,homem,1,0.5785,0\nreal_00762.jpg,0,homem,0,0.6012,1\nreal_00769.jpg,0,homem,0,0.9253,1\nreal_00815.jpg,0,homem,0,0.7846,1\nreal_00851.jpg,0,homem,0,0.8818,1\nreal_00895.jpg,0,homem,0,0.8431,1\nreal_00952.jpg,0,homem,0,0.8288,1\nreal_00954.jpg,0,homem,0,0.7567,1\nreal_00988.jpg,0,homem,0,0.5958,1\nreal_00990.jpg,0,homem,1,0.5236,0\nreal_01001.jpg,0,homem,0,0.7021,1\nreal_01004.jpg,0,homem,0,0.8591,1\nreal_01007.jpg,0,homem,0,0.5739,1\nreal_00002.jpg,0,mulher,1,0.8457,0\nreal_00010.jpg,0,mulher,0,0.8038,1\nreal_00028.jpg,0,mulher,0,0.5471,1\nreal_00044.jpg,0,mulher,0,0.6395,1\nreal_00054.jpg,0,mulher,1,0.659,0\nreal_00089.jpg,0,mulher,0,0.6391,1\nreal_00101.jpg,0,mulher,0,0.5207,1\nreal_00106.jpg,0,mulher,1,0.5484,0\nreal_00136.jpg,0,mulher,0,0.7497,1\nreal_00143.jpg,0,mulher,0,0.8412,1\nreal_00154.jpg,0,mulher,0,0.746,1\nreal_00155.jpg,0,mulher,0,0.8777,1\nreal_00164.jpg,0,mulher,1,0.7343,0\nreal_00167.jpg,0,mulher,1,0.5456,0\nreal_00212.jpg,0,mulher,0,0.7195,1\nreal_00216.jpg,0,mulher,1,0.7641,0\nreal_00257.jpg,0,mulher,0,0.5212,1\nreal_00271.jpg,0,mulher,0,0.5277,1\nreal_00275.jpg,0,mulher,0,0.8097,1\nreal_00290.jpg,0,mulher,0,0.7268,1\nreal_00293.jpg,0,mulher,0,0.5995,1\nreal_00304.jpg,0,mulher,1,0.5676,0\nreal_00312.jpg,0,mulher,1,0.577,0\nreal_00313.jpg,0,mulher,1,0.616,0\nreal_00316.jpg,0,mulher,0,0.8793,1\nreal_00319.jpg,0,mulher,0,0.877,1\nreal_00321.jpg,0,mulher,0,0.7783,1\nreal_00325.jpg,0,mulher,0,0.7786,1\nreal_00329.jpg,0,mulher,0,0.5274,1\nreal_00331.jpg,0,mulher,0,0.6144,1\nreal_00333.jpg,0,mulher,1,0.6003,0\nreal_00341.jpg,0,mulher,0,0.9105,1\nreal_00342.jpg,0,mulher,0,0.5127,1\nreal_00343.jpg,0,mulher,0,0.6053,1\nreal_00352.jpg,0,mulher,1,0.7582,0\nreal_00354.jpg,0,mulher,0,0.5162,1\nreal_00356.jpg,0,mulher,1,0.5429,0\nreal_00369.jpg,0,mulher,0,0.8661,1\nreal_00373.jpg,0,mulher,0,0.8004,1\nreal_00434.jpg,0,mulher,0,0.5222,1\nreal_00436.jpg,0,mulher,1,0.5311,0\nreal_00437.jpg,0,mulher,0,0.622,1\nreal_00443.jpg,0,mulher,0,0.6874,1\nreal_00456.jpg,0,mulher,1,0.6118,0\nreal_00463.jpg,0,mulher,0,0.8991,1\nreal_00474.jpg,0,mulher,0,0.5806,1\nreal_00478.jpg,0,mulher,0,0.8012,1\nreal_00485.jpg,0,mulher,0,0.8045,1\nreal_00493.jpg,0,mulher,0,0.7722,1\nreal_00510.jpg,0,mulher,0,0.7944,1\nreal_00512.jpg,0,mulher,0,0.7789,1\nreal_00533.jpg,0,mulher,1,0.6798,0\nreal_00548.jpg,0,mulher,1,0.7772,0\nreal_00551.jpg,0,mulher,1,0.8015,0\nreal_00617.jpg,0,mulher,1,0.8005,0\nreal_00620.jpg,0,mulher,1,0.5911,0\nreal_00634.jpg,0,mulher,1,0.668,0\nreal_00639.jpg,0,mulher,0,0.9069,1\nreal_00651.jpg,0,mulher,0,0.6832,1\nreal_00688.jpg,0,mulher,0,0.804,1\nreal_00697.jpg,0,mulher,0,0.9392,1\nreal_00707.jpg,0,mulher,1,0.5193,0\nreal_00720.jpg,0,mulher,0,0.6313,1\nreal_00725.jpg,0,mulher,1,0.5566,0\nreal_00754.jpg,0,mulher,0,0.7409,1\nreal_00760.jpg,0,mulher,1,0.6879,0\nreal_00767.jpg,0,mulher,0,0.5636,1\nreal_00774.jpg,0,mulher,1,0.7179,0\nreal_00811.jpg,0,mulher,0,0.5555,1\nreal_00813.jpg,0,mulher,0,0.5314,1\nreal_00817.jpg,0,mulher,1,0.9139,0\nreal_00829.jpg,0,mulher,1,0.7312,0\nreal_00837.jpg,0,mulher,0,0.5996,1\nreal_00839.jpg,0,mulher,1,0.8823,0\nreal_00843.jpg,0,mulher,1,0.5851,0\nreal_00858.jpg,0,mulher,1,0.5971,0\nreal_00872.jpg,0,mulher,0,0.5821,1\nreal_00879.jpg,0,mulher,0,0.5482,1\nreal_00882.jpg,0,mulher,1,0.786,0\nreal_00888.jpg,0,mulher,0,0.8161,1\nreal_00903.jpg,0,mulher,1,0.7313,0\nreal_00931.jpg,0,mulher,1,0.6595,0\nreal_00936.jpg,0,mulher,0,0.6728,1\nreal_00938.jpg,0,mulher,1,0.7441,0\nreal_00975.jpg,0,mulher,1,0.5402,0\nreal_00981.jpg,0,mulher,0,0.7116,1\nreal_00982.jpg,0,mulher,0,0.791,1\nreal_00991.jpg,0,mulher,0,0.6563,1\nreal_00992.jpg,0,mulher,0,0.8111,1\nreal_01026.jpg,0,mulher,0,0.7883,1\nreal_01031.jpg,0,mulher,1,0.6112,0\nreal_01038.jpg,0,mulher,0,0.6666,1\nreal_01046.jpg,0,mulher,0,0.8648,1\nreal_01073.jpg,0,mulher,0,0.7882,1\neasy_134_0001.jpg,1,homem,0,0.8217,0\neasy_136_0111.jpg,1,homem,1,0.5212,1\neasy_142_1111.jpg,1,homem,0,0.5522,0\neasy_148_0011.jpg,1,homem,0,0.8871,0\neasy_164_1111.jpg,1,homem,0,0.9351,0\neasy_216_1100.jpg,1,homem,1,0.5409,1\neasy_221_0111.jpg,1,homem,0,0.5043,0\neasy_31_0011.jpg,1,homem,1,0.6745,1\neasy_47_1110.jpg,1,homem,0,0.6857,0\neasy_61_0111.jpg,1,homem,0,0.6918,0\neasy_85_1111.jpg,1,homem,1,0.8722,1\nhard_111_1111.jpg,1,homem,1,0.7555,1\nhard_11_1100.jpg,1,homem,1,0.8123,1\nhard_131_1000.jpg,1,homem,0,0.6986,0\nhard_134_1100.jpg,1,homem,0,0.727,0\nhard_143_0100.jpg,1,homem,1,0.7105,1\nhard_146_0110.jpg,1,homem,0,0.5826,0\nhard_156_1111.jpg,1,homem,1,0.5527,1\nhard_175_1100.jpg,1,homem,1,0.6433,1\nhard_186_0011.jpg,1,homem,1,0.6079,1\nhard_195_1000.jpg,1,homem,1,0.7264,1\nhard_20_1111.jpg,1,homem,1,0.5821,1\nhard_235_0001.jpg,1,homem,1,0.5499,1\nhard_239_1000.jpg,1,homem,0,0.5071,0\nhard_28_1111.jpg,1,homem,1,0.8311,1\nhard_77_1100.jpg,1,homem,0,0.5831,0\nhard_78_1101.jpg,1,homem,0,0.8267,0\nhard_86_1100.jpg,1,homem,1,0.6184,1\nmid_122_1100.jpg,1,homem,0,0.5264,0\nmid_129_0101.jpg,1,homem,1,0.662,1\nmid_136_0001.jpg,1,homem,1,0.7072,1\nmid_164_0011.jpg,1,homem,1,0.8308,1\nmid_173_1110.jpg,1,homem,1,0.6366,1\nmid_17_1111.jpg,1,homem,0,0.5216,0\nmid_196_1100.jpg,1,homem,1,0.832,1\nmid_202_1110.jpg,1,homem,0,0.5215,0\nmid_240_1100.jpg,1,homem,1,0.5899,1\nmid_292_1000.jpg,1,homem,0,0.5414,0\nmid_299_1100.jpg,1,homem,1,0.7824,1\nmid_29_1111.jpg,1,homem,1,0.8092,1\nmid_332_1111.jpg,1,homem,1,0.5793,1\nmid_335_1100.jpg,1,homem,0,0.6278,0\nmid_337_1111.jpg,1,homem,0,0.5759,0\nmid_344_0010.jpg,1,homem,0,0.5491,0\nmid_364_1111.jpg,1,homem,1,0.5828,1\nmid_369_1110.jpg,1,homem,1,0.6559,1\nmid_371_0011.jpg,1,homem,1,0.6584,1\nmid_376_0001.jpg,1,homem,1,0.6692,1\nmid_377_0100.jpg,1,homem,1,0.693,1\nmid_391_1110.jpg,1,homem,1,0.7617,1\nmid_392_1111.jpg,1,homem,0,0.8265,0\nmid_423_1101.jpg,1,homem,0,0.5284,0\nmid_433_0011.jpg,1,homem,1,0.7151,1\nmid_434_1110.jpg,1,homem,0,0.6375,0\nmid_461_1100.jpg,1,homem,0,0.6557,0\nmid_465_1111.jpg,1,homem,0,0.813,0\nmid_50_1111.jpg,1,homem,0,0.6716,0\nmid_53_1100.jpg,1,homem,1,0.6764,1\nmid_68_1111.jpg,1,homem,0,0.723,0\neasy_115_0010.jpg,1,mulher,0,0.5236,0\neasy_129_1111.jpg,1,mulher,1,0.5177,1\neasy_132_0110.jpg,1,mulher,1,0.6777,1\neasy_151_1110.jpg,1,mulher,0,0.6178,0\neasy_166_1100.jpg,1,mulher,0,0.5035,0\neasy_172_1001.jpg,1,mulher,0,0.5964,0\neasy_173_0111.jpg,1,mulher,0,0.6248,0\neasy_180_0011.jpg,1,mulher,0,0.77,0\neasy_182_0100.jpg,1,mulher,1,0.7784,1\neasy_188_1111.jpg,1,mulher,0,0.6301,0\neasy_18_0011.jpg,1,mulher,0,0.522,0\neasy_208_1111.jpg,1,mulher,1,0.5036,1\neasy_224_1100.jpg,1,mulher,0,0.6153,0\neasy_234_1000.jpg,1,mulher,1,0.7736,1\neasy_236_1100.jpg,1,mulher,1,0.6936,1\neasy_90_1111.jpg,1,mulher,0,0.6469,0\neasy_91_1000.jpg,1,mulher,0,0.7906,0\neasy_92_0011.jpg,1,mulher,1,0.7101,1\nhard_102_1001.jpg,1,mulher,1,0.719,1\nhard_108_1111.jpg,1,mulher,0,0.6789,0\nhard_123_1111.jpg,1,mulher,1,0.61,1\nhard_145_1100.jpg,1,mulher,0,0.6522,0\nhard_152_0111.jpg,1,mulher,0,0.5092,0\nhard_170_0111.jpg,1,mulher,0,0.7717,0\nhard_18_1111.jpg,1,mulher,0,0.6088,0\nhard_214_1011.jpg,1,mulher,1,0.7747,1\nhard_218_0100.jpg,1,mulher,1,0.6979,1\nhard_21_1101.jpg,1,mulher,1,0.7597,1\nhard_231_1111.jpg,1,mulher,0,0.7912,0\nhard_24_1100.jpg,1,mulher,1,0.7258,1\nhard_48_0011.jpg,1,mulher,1,0.6566,1\nhard_63_1101.jpg,1,mulher,1,0.6652,1\nhard_80_0100.jpg,1,mulher,1,0.5259,1\nhard_87_1111.jpg,1,mulher,1,0.6816,1\nhard_93_1111.jpg,1,mulher,1,0.5953,1\nhard_99_1111.jpg,1,mulher,1,0.5248,1\nmid_108_1110.jpg,1,mulher,1,0.8391,1\nmid_10_1111.jpg,1,mulher,1,0.8366,1\nmid_11_1100.jpg,1,mulher,1,0.697,1\nmid_133_1001.jpg,1,mulher,1,0.7316,1\nmid_134_1010.jpg,1,mulher,0,0.6185,0\nmid_140_1111.jpg,1,mulher,0,0.708,0\nmid_141_1110.jpg,1,mulher,1,0.763,1\nmid_145_1111.jpg,1,mulher,1,0.5734,1\nmid_147_1110.jpg,1,mulher,1,0.7561,1\nmid_149_0011.jpg,1,mulher,0,0.6123,0\nmid_14_1101.jpg,1,mulher,1,0.6768,1\nmid_151_1100.jpg,1,mulher,1,0.5426,1\nmid_158_1010.jpg,1,mulher,1,0.5306,1\nmid_159_1111.jpg,1,mulher,1,0.523,1\nmid_179_1111.jpg,1,mulher,1,0.8896,1\nmid_184_1101.jpg,1,mulher,1,0.8372,1\nmid_220_0011.jpg,1,mulher,0,0.732,0\nmid_231_1100.jpg,1,mulher,0,0.5558,0\nmid_241_1111.jpg,1,mulher,1,0.5654,1\nmid_244_1110.jpg,1,mulher,0,0.5922,0\nmid_248_0111.jpg,1,mulher,1,0.7108,1\nmid_251_1111.jpg,1,mulher,1,0.8193,1\nmid_267_1111.jpg,1,mulher,1,0.707,1\nmid_273_1101.jpg,1,mulher,1,0.6865,1\nmid_279_1111.jpg,1,mulher,1,0.7009,1\nmid_306_1111.jpg,1,mulher,1,0.7101,1\nmid_327_0111.jpg,1,mulher,1,0.6914,1\nmid_339_1100.jpg,1,mulher,1,0.7185,1\nmid_33_1011.jpg,1,mulher,1,0.862,1\nmid_345_1111.jpg,1,mulher,1,0.6259,1\nmid_347_0011.jpg,1,mulher,1,0.5421,1\nmid_362_1101.jpg,1,mulher,0,0.831,0\nmid_378_1110.jpg,1,mulher,1,0.7124,1\nmid_380_1111.jpg,1,mulher,0,0.5249,0\nmid_389_1101.jpg,1,mulher,1,0.7575,1\nmid_393_1101.jpg,1,mulher,1,0.9086,1\nmid_395_1110.jpg,1,mulher,0,0.5205,0\nmid_419_0011.jpg,1,mulher,0,0.6996,0\nmid_422_0011.jpg,1,mulher,1,0.8519,1\nmid_427_1010.jpg,1,mulher,0,0.5211,0\nmid_431_0001.jpg,1,mulher,1,0.722,1\nmid_443_0011.jpg,1,mulher,1,0.7396,1\nmid_470_1001.jpg,1,mulher,1,0.7155,1\nmid_477_1100.jpg,1,mulher,0,0.6929,0\nmid_59_1111.jpg,1,mulher,1,0.7111,1\nmid_65_1111.jpg,1,mulher,1,0.6453,1\nmid_70_1111.jpg,1,mulher,1,0.6089,1\nmid_78_1101.jpg,1,mulher,1,0.6018,1\nmid_99_1100.jpg,1,mulher,0,0.7491,0\n'))
adv=pd.read_csv(io.StringIO('filename,label,gender,pred_limpa,conf_limpa,pred_adversarial,conf_adversarial,ataque_bem_sucedido\nreal_00031.jpg,0,homem,0,0.594,1,0.8827,1\nreal_00037.jpg,0,homem,1,0.5462,1,0.9414,0\nreal_00038.jpg,0,homem,1,0.7542,1,0.9854,0\nreal_00060.jpg,0,homem,0,0.8344,1,0.5757,1\nreal_00072.jpg,0,homem,1,0.5983,1,0.8347,0\nreal_00087.jpg,0,homem,0,0.5529,1,0.8058,1\nreal_00090.jpg,0,homem,0,0.5409,1,0.8335,1\nreal_00093.jpg,0,homem,0,0.5861,1,0.9614,1\nreal_00094.jpg,0,homem,0,0.5058,1,0.8304,1\nreal_00099.jpg,0,homem,0,0.9462,1,0.5611,1\nreal_00105.jpg,0,homem,0,0.7679,1,0.6494,1\nreal_00159.jpg,0,homem,0,0.5567,1,0.6763,1\nreal_00162.jpg,0,homem,0,0.5674,1,0.8412,1\nreal_00178.jpg,0,homem,1,0.5459,1,0.9737,0\nreal_00179.jpg,0,homem,0,0.6279,1,0.7821,1\nreal_00227.jpg,0,homem,0,0.8569,1,0.5498,1\nreal_00237.jpg,0,homem,0,0.6019,1,0.8472,1\nreal_00254.jpg,0,homem,0,0.5807,1,0.5979,1\nreal_00259.jpg,0,homem,0,0.7274,1,0.6981,1\nreal_00264.jpg,0,homem,1,0.6782,1,0.8531,0\nreal_00390.jpg,0,homem,0,0.8234,1,0.8757,1\nreal_00392.jpg,0,homem,1,0.5551,1,0.8681,0\nreal_00400.jpg,0,homem,0,0.7126,1,0.9257,1\nreal_00406.jpg,0,homem,0,0.7204,1,0.6657,1\nreal_00412.jpg,0,homem,0,0.8546,1,0.7665,1\nreal_00418.jpg,0,homem,0,0.7092,1,0.6448,1\nreal_00453.jpg,0,homem,0,0.5273,1,0.6272,1\nreal_00459.jpg,0,homem,0,0.7855,0,0.5304,0\nreal_00462.jpg,0,homem,0,0.8317,1,0.9086,1\nreal_00492.jpg,0,homem,0,0.83,0,0.5414,0\nreal_00500.jpg,0,homem,0,0.8438,0,0.6017,0\nreal_00506.jpg,0,homem,0,0.843,1,0.5971,1\nreal_00530.jpg,0,homem,0,0.7483,1,0.8998,1\nreal_00532.jpg,0,homem,0,0.6882,0,0.5951,0\nreal_00534.jpg,0,homem,1,0.6268,1,0.9039,0\nreal_00535.jpg,0,homem,1,0.8855,1,0.9033,0\nreal_00556.jpg,0,homem,0,0.5868,1,0.8634,1\nreal_00566.jpg,0,homem,0,0.8142,1,0.783,1\nreal_00571.jpg,0,homem,1,0.7831,1,0.8917,0\nreal_00582.jpg,0,homem,0,0.6571,1,0.7396,1\nreal_00590.jpg,0,homem,0,0.6732,1,0.8225,1\nreal_00594.jpg,0,homem,1,0.5571,1,0.8218,0\nreal_00598.jpg,0,homem,0,0.7881,1,0.7295,1\nreal_00604.jpg,0,homem,0,0.6043,1,0.7756,1\nreal_00605.jpg,0,homem,0,0.7259,1,0.8049,1\nreal_00610.jpg,0,homem,0,0.7961,1,0.8069,1\nreal_00623.jpg,0,homem,0,0.737,1,0.5714,1\nreal_00654.jpg,0,homem,0,0.7549,1,0.7065,1\nreal_00666.jpg,0,homem,0,0.7031,1,0.8999,1\nreal_00674.jpg,0,homem,0,0.5341,1,0.8801,1\nreal_00686.jpg,0,homem,0,0.8071,1,0.6476,1\nreal_00698.jpg,0,homem,0,0.6597,1,0.9479,1\nreal_00702.jpg,0,homem,0,0.6199,1,0.5653,1\nreal_00706.jpg,0,homem,0,0.8875,1,0.6693,1\nreal_00722.jpg,0,homem,0,0.8288,1,0.5376,1\nreal_00741.jpg,0,homem,1,0.5743,1,0.9575,0\nreal_00756.jpg,0,homem,1,0.5785,1,0.8306,0\nreal_00762.jpg,0,homem,0,0.6012,1,0.7581,1\nreal_00769.jpg,0,homem,0,0.9253,1,0.8041,1\nreal_00815.jpg,0,homem,0,0.7846,1,0.5482,1\nreal_00851.jpg,0,homem,0,0.8818,0,0.6175,0\nreal_00895.jpg,0,homem,0,0.8431,1,0.8873,1\nreal_00952.jpg,0,homem,0,0.8288,1,0.6885,1\nreal_00954.jpg,0,homem,0,0.7567,1,0.7102,1\nreal_00988.jpg,0,homem,0,0.5958,1,0.6616,1\nreal_00990.jpg,0,homem,1,0.5236,1,0.849,0\nreal_01001.jpg,0,homem,0,0.7021,1,0.6066,1\nreal_01004.jpg,0,homem,0,0.8591,1,0.8893,1\nreal_01007.jpg,0,homem,0,0.5739,1,0.9151,1\nreal_00002.jpg,0,mulher,1,0.8457,1,0.9462,0\nreal_00010.jpg,0,mulher,0,0.8038,1,0.7647,1\nreal_00028.jpg,0,mulher,0,0.5471,1,0.903,1\nreal_00044.jpg,0,mulher,0,0.6395,1,0.8384,1\nreal_00054.jpg,0,mulher,1,0.659,1,0.9007,0\nreal_00089.jpg,0,mulher,0,0.6391,1,0.5868,1\nreal_00101.jpg,0,mulher,0,0.5207,1,0.8625,1\nreal_00106.jpg,0,mulher,1,0.5484,1,0.8716,0\nreal_00136.jpg,0,mulher,0,0.7497,1,0.8475,1\nreal_00143.jpg,0,mulher,0,0.8412,1,0.6281,1\nreal_00154.jpg,0,mulher,0,0.746,1,0.8359,1\nreal_00155.jpg,0,mulher,0,0.8777,0,0.5997,0\nreal_00164.jpg,0,mulher,1,0.7343,1,0.9843,0\nreal_00167.jpg,0,mulher,1,0.5456,1,0.7444,0\nreal_00212.jpg,0,mulher,0,0.7195,1,0.7778,1\nreal_00216.jpg,0,mulher,1,0.7641,1,0.9541,0\nreal_00257.jpg,0,mulher,0,0.5212,1,0.9343,1\nreal_00271.jpg,0,mulher,0,0.5277,1,0.9043,1\nreal_00275.jpg,0,mulher,0,0.8097,0,0.5013,0\nreal_00290.jpg,0,mulher,0,0.7268,1,0.82,1\nreal_00293.jpg,0,mulher,0,0.5995,1,0.8832,1\nreal_00304.jpg,0,mulher,1,0.5676,1,0.9058,0\nreal_00312.jpg,0,mulher,1,0.577,1,0.9678,0\nreal_00313.jpg,0,mulher,1,0.616,1,0.9141,0\nreal_00316.jpg,0,mulher,0,0.8793,1,0.5866,1\nreal_00319.jpg,0,mulher,0,0.877,0,0.5242,0\nreal_00321.jpg,0,mulher,0,0.7783,1,0.8688,1\nreal_00325.jpg,0,mulher,0,0.7786,0,0.5121,0\nreal_00329.jpg,0,mulher,0,0.5274,1,0.9588,1\nreal_00331.jpg,0,mulher,0,0.6144,1,0.8362,1\nreal_00333.jpg,0,mulher,1,0.6003,1,0.8024,0\nreal_00341.jpg,0,mulher,0,0.9105,0,0.613,0\nreal_00342.jpg,0,mulher,0,0.5127,1,0.9274,1\nreal_00343.jpg,0,mulher,0,0.6053,1,0.7755,1\nreal_00352.jpg,0,mulher,1,0.7582,1,0.9394,0\nreal_00354.jpg,0,mulher,0,0.5162,1,0.9195,1\nreal_00356.jpg,0,mulher,1,0.5429,1,0.8419,0\nreal_00369.jpg,0,mulher,0,0.8661,1,0.6719,1\nreal_00373.jpg,0,mulher,0,0.8004,1,0.9155,1\nreal_00434.jpg,0,mulher,0,0.5222,1,0.9374,1\nreal_00436.jpg,0,mulher,1,0.5311,1,0.7464,0\nreal_00437.jpg,0,mulher,0,0.622,1,0.9711,1\nreal_00443.jpg,0,mulher,0,0.6874,1,0.6964,1\nreal_00456.jpg,0,mulher,1,0.6118,1,0.9599,0\nreal_00463.jpg,0,mulher,0,0.8991,1,0.8706,1\nreal_00474.jpg,0,mulher,0,0.5806,1,0.9386,1\nreal_00478.jpg,0,mulher,0,0.8012,1,0.8001,1\nreal_00485.jpg,0,mulher,0,0.8045,1,0.5489,1\nreal_00493.jpg,0,mulher,0,0.7722,1,0.5301,1\nreal_00510.jpg,0,mulher,0,0.7944,1,0.6232,1\nreal_00512.jpg,0,mulher,0,0.7789,1,0.7031,1\nreal_00533.jpg,0,mulher,1,0.6798,1,0.896,0\nreal_00548.jpg,0,mulher,1,0.7772,1,0.9531,0\nreal_00551.jpg,0,mulher,1,0.8015,1,0.8911,0\nreal_00617.jpg,0,mulher,1,0.8005,1,0.8723,0\nreal_00620.jpg,0,mulher,1,0.5911,1,0.9627,0\nreal_00634.jpg,0,mulher,1,0.668,1,0.9201,0\nreal_00639.jpg,0,mulher,0,0.9069,1,0.6589,1\nreal_00651.jpg,0,mulher,0,0.6832,1,0.9072,1\nreal_00688.jpg,0,mulher,0,0.804,1,0.6787,1\nreal_00697.jpg,0,mulher,0,0.9392,0,0.6449,0\nreal_00707.jpg,0,mulher,1,0.5193,1,0.8675,0\nreal_00720.jpg,0,mulher,0,0.6313,1,0.7821,1\nreal_00725.jpg,0,mulher,1,0.5566,1,0.9614,0\nreal_00754.jpg,0,mulher,0,0.7409,1,0.7172,1\nreal_00760.jpg,0,mulher,1,0.6879,1,0.9756,0\nreal_00767.jpg,0,mulher,0,0.5636,1,0.973,1\nreal_00774.jpg,0,mulher,1,0.7179,1,0.9769,0\nreal_00811.jpg,0,mulher,0,0.5555,1,0.9524,1\nreal_00813.jpg,0,mulher,0,0.5314,1,0.7766,1\nreal_00817.jpg,0,mulher,1,0.9139,1,0.9585,0\nreal_00829.jpg,0,mulher,1,0.7312,1,0.8128,0\nreal_00837.jpg,0,mulher,0,0.5996,1,0.891,1\nreal_00839.jpg,0,mulher,1,0.8823,1,0.9357,0\nreal_00843.jpg,0,mulher,1,0.5851,1,0.9034,0\nreal_00858.jpg,0,mulher,1,0.5971,1,0.9401,0\nreal_00872.jpg,0,mulher,0,0.5821,1,0.7749,1\nreal_00879.jpg,0,mulher,0,0.5482,1,0.9667,1\nreal_00882.jpg,0,mulher,1,0.786,1,0.9313,0\nreal_00888.jpg,0,mulher,0,0.8161,1,0.5949,1\nreal_00903.jpg,0,mulher,1,0.7313,1,0.8797,0\nreal_00931.jpg,0,mulher,1,0.6595,1,0.9657,0\nreal_00936.jpg,0,mulher,0,0.6728,1,0.8809,1\nreal_00938.jpg,0,mulher,1,0.7441,1,0.9518,0\nreal_00975.jpg,0,mulher,1,0.5402,1,0.9307,0\nreal_00981.jpg,0,mulher,0,0.7116,1,0.7618,1\nreal_00982.jpg,0,mulher,0,0.791,1,0.6016,1\nreal_00991.jpg,0,mulher,0,0.6563,1,0.7245,1\nreal_00992.jpg,0,mulher,0,0.8111,0,0.6194,0\nreal_01026.jpg,0,mulher,0,0.7883,1,0.7838,1\nreal_01031.jpg,0,mulher,1,0.6112,1,0.9814,0\nreal_01038.jpg,0,mulher,0,0.6666,1,0.6637,1\nreal_01046.jpg,0,mulher,0,0.8648,1,0.8977,1\nreal_01073.jpg,0,mulher,0,0.7882,1,0.7497,1\neasy_134_0001.jpg,1,homem,0,0.8217,0,0.942,0\neasy_136_0111.jpg,1,homem,1,0.5212,0,0.8583,1\neasy_142_1111.jpg,1,homem,0,0.5522,0,0.9121,0\neasy_148_0011.jpg,1,homem,0,0.8871,0,0.9593,0\neasy_164_1111.jpg,1,homem,0,0.9351,0,0.9077,0\neasy_216_1100.jpg,1,homem,1,0.5409,0,0.8256,1\neasy_221_0111.jpg,1,homem,0,0.5043,0,0.8305,0\neasy_31_0011.jpg,1,homem,1,0.6745,0,0.9175,1\neasy_47_1110.jpg,1,homem,0,0.6857,0,0.9664,0\neasy_61_0111.jpg,1,homem,0,0.6918,0,0.8824,0\neasy_85_1111.jpg,1,homem,1,0.8722,0,0.7834,1\nhard_111_1111.jpg,1,homem,1,0.7555,0,0.723,1\nhard_11_1100.jpg,1,homem,1,0.8123,0,0.7182,1\nhard_131_1000.jpg,1,homem,0,0.6986,0,0.965,0\nhard_134_1100.jpg,1,homem,0,0.727,0,0.971,0\nhard_143_0100.jpg,1,homem,1,0.7105,0,0.9332,1\nhard_146_0110.jpg,1,homem,0,0.5826,0,0.9534,0\nhard_156_1111.jpg,1,homem,1,0.5527,0,0.9721,1\nhard_175_1100.jpg,1,homem,1,0.6433,0,0.8346,1\nhard_186_0011.jpg,1,homem,1,0.6079,0,0.8723,1\nhard_195_1000.jpg,1,homem,1,0.7264,0,0.7956,1\nhard_20_1111.jpg,1,homem,1,0.5821,0,0.8282,1\nhard_235_0001.jpg,1,homem,1,0.5499,0,0.9047,1\nhard_239_1000.jpg,1,homem,0,0.5071,0,0.9126,0\nhard_28_1111.jpg,1,homem,1,0.8311,0,0.7458,1\nhard_77_1100.jpg,1,homem,0,0.5831,0,0.8509,0\nhard_78_1101.jpg,1,homem,0,0.8267,0,0.9248,0\nhard_86_1100.jpg,1,homem,1,0.6184,0,0.9306,1\nmid_122_1100.jpg,1,homem,0,0.5264,0,0.9224,0\nmid_129_0101.jpg,1,homem,1,0.662,0,0.7237,1\nmid_136_0001.jpg,1,homem,1,0.7072,0,0.8895,1\nmid_164_0011.jpg,1,homem,1,0.8308,0,0.7568,1\nmid_173_1110.jpg,1,homem,1,0.6366,0,0.8006,1\nmid_17_1111.jpg,1,homem,0,0.5216,0,0.9542,0\nmid_196_1100.jpg,1,homem,1,0.832,0,0.7407,1\nmid_202_1110.jpg,1,homem,0,0.5215,0,0.9355,0\nmid_240_1100.jpg,1,homem,1,0.5899,0,0.7679,1\nmid_292_1000.jpg,1,homem,0,0.5414,0,0.9634,0\nmid_299_1100.jpg,1,homem,1,0.7824,0,0.8404,1\nmid_29_1111.jpg,1,homem,1,0.8092,0,0.6111,1\nmid_332_1111.jpg,1,homem,1,0.5793,0,0.8148,1\nmid_335_1100.jpg,1,homem,0,0.6278,0,0.8494,0\nmid_337_1111.jpg,1,homem,0,0.5759,0,0.9182,0\nmid_344_0010.jpg,1,homem,0,0.5491,0,0.9118,0\nmid_364_1111.jpg,1,homem,1,0.5828,0,0.7632,1\nmid_369_1110.jpg,1,homem,1,0.6559,0,0.8375,1\nmid_371_0011.jpg,1,homem,1,0.6584,0,0.9152,1\nmid_376_0001.jpg,1,homem,1,0.6692,0,0.9098,1\nmid_377_0100.jpg,1,homem,1,0.693,0,0.8691,1\nmid_391_1110.jpg,1,homem,1,0.7617,0,0.9288,1\nmid_392_1111.jpg,1,homem,0,0.8265,0,0.9495,0\nmid_423_1101.jpg,1,homem,0,0.5284,0,0.9438,0\nmid_433_0011.jpg,1,homem,1,0.7151,0,0.7871,1\nmid_434_1110.jpg,1,homem,0,0.6375,0,0.9209,0\nmid_461_1100.jpg,1,homem,0,0.6557,0,0.9642,0\nmid_465_1111.jpg,1,homem,0,0.813,0,0.9804,0\nmid_50_1111.jpg,1,homem,0,0.6716,0,0.9203,0\nmid_53_1100.jpg,1,homem,1,0.6764,0,0.9581,1\nmid_68_1111.jpg,1,homem,0,0.723,0,0.9697,0\neasy_115_0010.jpg,1,mulher,0,0.5236,0,0.9026,0\neasy_129_1111.jpg,1,mulher,1,0.5177,0,0.9444,1\neasy_132_0110.jpg,1,mulher,1,0.6777,0,0.8572,1\neasy_151_1110.jpg,1,mulher,0,0.6178,0,0.9206,0\neasy_166_1100.jpg,1,mulher,0,0.5035,0,0.9277,0\neasy_172_1001.jpg,1,mulher,0,0.5964,0,0.9445,0\neasy_173_0111.jpg,1,mulher,0,0.6248,0,0.8701,0\neasy_180_0011.jpg,1,mulher,0,0.77,0,0.9579,0\neasy_182_0100.jpg,1,mulher,1,0.7784,0,0.9637,1\neasy_188_1111.jpg,1,mulher,0,0.6301,0,0.9386,0\neasy_18_0011.jpg,1,mulher,0,0.522,0,0.9305,0\neasy_208_1111.jpg,1,mulher,1,0.5036,0,0.9235,1\neasy_224_1100.jpg,1,mulher,0,0.6153,0,0.9665,0\neasy_234_1000.jpg,1,mulher,1,0.7736,0,0.8396,1\neasy_236_1100.jpg,1,mulher,1,0.6936,0,0.9132,1\neasy_90_1111.jpg,1,mulher,0,0.6469,0,0.9716,0\neasy_91_1000.jpg,1,mulher,0,0.7906,0,0.9487,0\neasy_92_0011.jpg,1,mulher,1,0.7101,0,0.9059,1\nhard_102_1001.jpg,1,mulher,1,0.719,0,0.8329,1\nhard_108_1111.jpg,1,mulher,0,0.6789,0,0.8805,0\nhard_123_1111.jpg,1,mulher,1,0.61,0,0.946,1\nhard_145_1100.jpg,1,mulher,0,0.6522,0,0.9566,0\nhard_152_0111.jpg,1,mulher,0,0.5092,0,0.8975,0\nhard_170_0111.jpg,1,mulher,0,0.7717,0,0.9693,0\nhard_18_1111.jpg,1,mulher,0,0.6088,0,0.9027,0\nhard_214_1011.jpg,1,mulher,1,0.7747,0,0.9016,1\nhard_218_0100.jpg,1,mulher,1,0.6979,0,0.8024,1\nhard_21_1101.jpg,1,mulher,1,0.7597,0,0.624,1\nhard_231_1111.jpg,1,mulher,0,0.7912,0,0.9692,0\nhard_24_1100.jpg,1,mulher,1,0.7258,0,0.7899,1\nhard_48_0011.jpg,1,mulher,1,0.6566,0,0.9607,1\nhard_63_1101.jpg,1,mulher,1,0.6652,0,0.9392,1\nhard_80_0100.jpg,1,mulher,1,0.5259,0,0.8947,1\nhard_87_1111.jpg,1,mulher,1,0.6816,0,0.7445,1\nhard_93_1111.jpg,1,mulher,1,0.5953,0,0.7064,1\nhard_99_1111.jpg,1,mulher,1,0.5248,0,0.9093,1\nmid_108_1110.jpg,1,mulher,1,0.8391,0,0.7421,1\nmid_10_1111.jpg,1,mulher,1,0.8366,0,0.8146,1\nmid_11_1100.jpg,1,mulher,1,0.697,0,0.8909,1\nmid_133_1001.jpg,1,mulher,1,0.7316,0,0.9308,1\nmid_134_1010.jpg,1,mulher,0,0.6185,0,0.9512,0\nmid_140_1111.jpg,1,mulher,0,0.708,0,0.9459,0\nmid_141_1110.jpg,1,mulher,1,0.763,0,0.9226,1\nmid_145_1111.jpg,1,mulher,1,0.5734,0,0.8496,1\nmid_147_1110.jpg,1,mulher,1,0.7561,0,0.9243,1\nmid_149_0011.jpg,1,mulher,0,0.6123,0,0.9566,0\nmid_14_1101.jpg,1,mulher,1,0.6768,0,0.8084,1\nmid_151_1100.jpg,1,mulher,1,0.5426,0,0.9526,1\nmid_158_1010.jpg,1,mulher,1,0.5306,0,0.932,1\nmid_159_1111.jpg,1,mulher,1,0.523,0,0.7348,1\nmid_179_1111.jpg,1,mulher,1,0.8896,0,0.7751,1\nmid_184_1101.jpg,1,mulher,1,0.8372,0,0.8499,1\nmid_220_0011.jpg,1,mulher,0,0.732,0,0.9756,0\nmid_231_1100.jpg,1,mulher,0,0.5558,0,0.9607,0\nmid_241_1111.jpg,1,mulher,1,0.5654,0,0.9154,1\nmid_244_1110.jpg,1,mulher,0,0.5922,0,0.9803,0\nmid_248_0111.jpg,1,mulher,1,0.7108,0,0.9585,1\nmid_251_1111.jpg,1,mulher,1,0.8193,0,0.7519,1\nmid_267_1111.jpg,1,mulher,1,0.707,0,0.777,1\nmid_273_1101.jpg,1,mulher,1,0.6865,0,0.8352,1\nmid_279_1111.jpg,1,mulher,1,0.7009,0,0.7969,1\nmid_306_1111.jpg,1,mulher,1,0.7101,0,0.9521,1\nmid_327_0111.jpg,1,mulher,1,0.6914,0,0.7634,1\nmid_339_1100.jpg,1,mulher,1,0.7185,0,0.8609,1\nmid_33_1011.jpg,1,mulher,1,0.862,0,0.7724,1\nmid_345_1111.jpg,1,mulher,1,0.6259,0,0.8708,1\nmid_347_0011.jpg,1,mulher,1,0.5421,0,0.8989,1\nmid_362_1101.jpg,1,mulher,0,0.831,0,0.9699,0\nmid_378_1110.jpg,1,mulher,1,0.7124,0,0.6891,1\nmid_380_1111.jpg,1,mulher,0,0.5249,0,0.8887,0\nmid_389_1101.jpg,1,mulher,1,0.7575,0,0.904,1\nmid_393_1101.jpg,1,mulher,1,0.9086,0,0.8796,1\nmid_395_1110.jpg,1,mulher,0,0.5205,0,0.9753,0\nmid_419_0011.jpg,1,mulher,0,0.6996,0,0.9688,0\nmid_422_0011.jpg,1,mulher,1,0.8519,0,0.854,1\nmid_427_1010.jpg,1,mulher,0,0.5211,0,0.8773,0\nmid_431_0001.jpg,1,mulher,1,0.722,0,0.9244,1\nmid_443_0011.jpg,1,mulher,1,0.7396,0,0.9569,1\nmid_470_1001.jpg,1,mulher,1,0.7155,0,0.7422,1\nmid_477_1100.jpg,1,mulher,0,0.6929,0,0.9621,0\nmid_59_1111.jpg,1,mulher,1,0.7111,0,0.905,1\nmid_65_1111.jpg,1,mulher,1,0.6453,0,0.7532,1\nmid_70_1111.jpg,1,mulher,1,0.6089,0,0.9589,1\nmid_78_1101.jpg,1,mulher,1,0.6018,0,0.746,1\nmid_99_1100.jpg,1,mulher,0,0.7491,0,0.9795,0\n'))
split=pd.read_csv(io.StringIO('filename,label,gender,split\nreal_00358.jpg,0,mulher,train\nreal_00701.jpg,0,homem,train\nreal_00961.jpg,0,homem,train\nreal_00862.jpg,0,homem,train\nreal_00248.jpg,0,mulher,train\nreal_00905.jpg,0,homem,train\nreal_00370.jpg,0,mulher,train\nreal_00915.jpg,0,mulher,train\nreal_00511.jpg,0,homem,train\nmid_331_0011.jpg,1,mulher,train\nmid_260_1111.jpg,1,mulher,train\nreal_00340.jpg,0,mulher,train\neasy_220_1000.jpg,1,homem,train\nreal_00708.jpg,0,mulher,train\nmid_451_1111.jpg,1,homem,train\neasy_24_0011.jpg,1,mulher,train\nmid_119_1111.jpg,1,mulher,train\nhard_154_1100.jpg,1,mulher,train\nhard_209_1011.jpg,1,mulher,train\neasy_128_0111.jpg,1,mulher,train\nreal_00546.jpg,0,homem,train\nmid_256_1100.jpg,1,mulher,train\nreal_00376.jpg,0,mulher,train\nreal_00723.jpg,0,homem,train\nmid_308_1111.jpg,1,homem,train\nmid_207_1111.jpg,1,mulher,train\nmid_25_1111.jpg,1,mulher,train\nhard_198_1100.jpg,1,homem,train\nhard_162_0010.jpg,1,mulher,train\nmid_263_1100.jpg,1,homem,train\nmid_333_1101.jpg,1,mulher,train\nmid_343_1111.jpg,1,homem,train\nmid_152_1100.jpg,1,homem,train\nreal_00169.jpg,0,homem,train\nmid_384_0011.jpg,1,mulher,train\nreal_00121.jpg,0,homem,train\neasy_191_0001.jpg,1,homem,train\nreal_00846.jpg,0,homem,train\nreal_00077.jpg,0,homem,train\nreal_01068.jpg,0,homem,train\nmid_208_1110.jpg,1,homem,train\nreal_00524.jpg,0,mulher,train\nhard_140_1111.jpg,1,homem,train\nreal_00726.jpg,0,homem,train\nmid_94_1111.jpg,1,mulher,train\nreal_00966.jpg,0,homem,train\nreal_01059.jpg,0,homem,train\nhard_223_1111.jpg,1,mulher,train\neasy_93_1010.jpg,1,mulher,train\nmid_58_1100.jpg,1,homem,train\nreal_00367.jpg,0,mulher,train\nhard_226_0100.jpg,1,mulher,train\nmid_225_0111.jpg,1,mulher,train\nreal_00629.jpg,0,mulher,train\nmid_31_1111.jpg,1,mulher,train\neasy_157_1110.jpg,1,mulher,train\nmid_314_1101.jpg,1,homem,train\nreal_00402.jpg,0,homem,train\nreal_01051.jpg,0,mulher,train\nmid_447_0011.jpg,1,homem,train\nhard_137_1001.jpg,1,homem,train\nmid_118_1000.jpg,1,mulher,train\nreal_00011.jpg,0,mulher,train\nreal_00849.jpg,0,mulher,train\nreal_00519.jpg,0,mulher,train\nhard_199_0011.jpg,1,mulher,train\nmid_466_1111.jpg,1,homem,train\nreal_00752.jpg,0,mulher,train\nreal_00980.jpg,0,homem,train\nmid_125_1111.jpg,1,homem,train\nreal_00124.jpg,0,homem,train\neasy_193_1110.jpg,1,homem,train\nhard_230_1110.jpg,1,mulher,train\nhard_148_1010.jpg,1,mulher,train\nreal_00397.jpg,0,homem,train\nreal_00311.jpg,0,mulher,train\nreal_00049.jpg,0,mulher,train\nreal_00647.jpg,0,homem,train\nreal_00151.jpg,0,mulher,train\nhard_2_1111.jpg,1,mulher,train\nreal_00016.jpg,0,mulher,train\nmid_6_1111.jpg,1,homem,train\neasy_155_0011.jpg,1,mulher,train\nmid_36_1100.jpg,1,mulher,train\nreal_00071.jpg,0,homem,train\nmid_420_1101.jpg,1,homem,train\nreal_00209.jpg,0,homem,train\nreal_00258.jpg,0,mulher,train\nreal_01063.jpg,0,homem,train\neasy_159_1111.jpg,1,homem,train\nhard_150_1000.jpg,1,mulher,train\nmid_195_1111.jpg,1,mulher,train\neasy_105_1100.jpg,1,mulher,train\nreal_00863.jpg,0,homem,train\nmid_274_1100.jpg,1,mulher,train\neasy_185_1101.jpg,1,mulher,train\nhard_166_1110.jpg,1,homem,train\nreal_01000.jpg,0,homem,train\nreal_00637.jpg,0,mulher,train\nreal_00986.jpg,0,homem,train\nreal_00517.jpg,0,mulher,train\nreal_00372.jpg,0,mulher,train\nhard_133_0001.jpg,1,homem,train\neasy_21_0011.jpg,1,mulher,train\nmid_448_0011.jpg,1,mulher,train\nreal_00597.jpg,0,mulher,train\nhard_3_1111.jpg,1,mulher,train\nreal_00229.jpg,0,mulher,train\nhard_128_1111.jpg,1,homem,train\nreal_00656.jpg,0,homem,train\neasy_197_1111.jpg,1,homem,train\nreal_00095.jpg,0,mulher,train\nreal_00460.jpg,0,mulher,train\nreal_00252.jpg,0,mulher,train\nreal_00482.jpg,0,homem,train\nreal_00502.jpg,0,mulher,train\nreal_00389.jpg,0,homem,train\nreal_00557.jpg,0,homem,train\nreal_00578.jpg,0,homem,train\nhard_68_1100.jpg,1,mulher,train\nreal_00055.jpg,0,homem,train\nmid_174_1111.jpg,1,mulher,train\nhard_135_1101.jpg,1,homem,train\nreal_00161.jpg,0,mulher,train\nreal_00068.jpg,0,homem,train\nhard_45_1111.jpg,1,mulher,train\nreal_00079.jpg,0,homem,train\nmid_111_0001.jpg,1,homem,train\neasy_42_0001.jpg,1,mulher,train\nreal_00051.jpg,0,homem,train\nreal_00107.jpg,0,mulher,train\nreal_00747.jpg,0,homem,train\nreal_00555.jpg,0,homem,train\nreal_00211.jpg,0,mulher,train\nhard_15_1111.jpg,1,homem,train\nreal_00887.jpg,0,mulher,train\nreal_00133.jpg,0,mulher,train\nreal_00461.jpg,0,mulher,train\nreal_00668.jpg,0,mulher,train\nreal_00195.jpg,0,mulher,train\nhard_105_1011.jpg,1,homem,train\nreal_00921.jpg,0,mulher,train\nreal_00703.jpg,0,homem,train\nreal_00671.jpg,0,homem,train\nreal_00292.jpg,0,mulher,train\nreal_00867.jpg,0,mulher,train\nreal_00096.jpg,0,homem,train\nhard_5_1111.jpg,1,mulher,train\neasy_73_1110.jpg,1,mulher,train\nreal_00131.jpg,0,mulher,train\nreal_00097.jpg,0,homem,train\nreal_00913.jpg,0,mulher,train\nhard_215_1101.jpg,1,homem,train\nreal_01041.jpg,0,mulher,train\neasy_25_0110.jpg,1,mulher,train\nmid_387_1100.jpg,1,mulher,train\nreal_00008.jpg,0,homem,train\nmid_142_1100.jpg,1,homem,train\nmid_35_1101.jpg,1,mulher,train\nreal_00466.jpg,0,mulher,train\nreal_00865.jpg,0,mulher,train\nmid_408_1111.jpg,1,homem,train\nmid_7_1100.jpg,1,homem,train\nreal_00515.jpg,0,homem,train\nreal_00925.jpg,0,homem,train\nmid_209_1111.jpg,1,homem,train\nreal_00371.jpg,0,homem,train\nreal_00664.jpg,0,homem,train\nmid_189_0011.jpg,1,mulher,train\nreal_00146.jpg,0,homem,train\nmid_103_1111.jpg,1,mulher,train\nhard_115_1110.jpg,1,homem,train\nmid_411_1111.jpg,1,homem,train\nmid_219_1111.jpg,1,homem,train\nhard_118_0111.jpg,1,mulher,train\neasy_11_1111.jpg,1,homem,train\nreal_00567.jpg,0,homem,train\nmid_226_1111.jpg,1,mulher,train\nreal_00109.jpg,0,mulher,train\nreal_00764.jpg,0,homem,train\nreal_00819.jpg,0,mulher,train\nreal_00890.jpg,0,mulher,train\nreal_00945.jpg,0,mulher,train\nreal_00814.jpg,0,mulher,train\nhard_35_1111.jpg,1,mulher,train\nmid_405_0011.jpg,1,mulher,train\nreal_00065.jpg,0,mulher,train\nreal_00593.jpg,0,homem,train\nmid_69_1111.jpg,1,homem,train\nmid_162_0011.jpg,1,mulher,train\nreal_00859.jpg,0,mulher,train\nreal_00559.jpg,0,homem,train\nreal_00775.jpg,0,mulher,train\nmid_188_1111.jpg,1,homem,train\nreal_00910.jpg,0,mulher,train\nreal_00115.jpg,0,homem,train\neasy_55_0011.jpg,1,homem,train\nreal_00278.jpg,0,mulher,train\nreal_00687.jpg,0,homem,train\neasy_89_0010.jpg,1,mulher,train\nmid_334_1111.jpg,1,homem,train\nreal_00454.jpg,0,mulher,train\nmid_259_1101.jpg,1,mulher,train\nmid_349_1111.jpg,1,mulher,train\nmid_396_1011.jpg,1,homem,train\nreal_00332.jpg,0,mulher,train\nreal_00632.jpg,0,mulher,train\neasy_41_0001.jpg,1,mulher,train\nmid_1_1101.jpg,1,mulher,train\nmid_2_1101.jpg,1,mulher,train\nreal_00027.jpg,0,mulher,train\nreal_00268.jpg,0,mulher,train\nhard_74_0011.jpg,1,mulher,train\nreal_00727.jpg,0,mulher,train\nreal_00147.jpg,0,mulher,train\nmid_468_1110.jpg,1,homem,train\nmid_328_0011.jpg,1,homem,train\nreal_00433.jpg,0,homem,train\nmid_120_1100.jpg,1,homem,train\neasy_168_0010.jpg,1,homem,train\nreal_00098.jpg,0,homem,train\nmid_261_1101.jpg,1,homem,train\nreal_01065.jpg,0,mulher,train\nreal_00569.jpg,0,homem,train\nreal_00074.jpg,0,homem,train\nreal_00835.jpg,0,homem,train\nreal_00256.jpg,0,mulher,train\nhard_91_1100.jpg,1,mulher,train\nmid_316_1111.jpg,1,mulher,train\nreal_00538.jpg,0,homem,train\nreal_00491.jpg,0,homem,train\nhard_222_1001.jpg,1,mulher,train\nreal_00496.jpg,0,mulher,train\nreal_00140.jpg,0,mulher,train\nreal_00221.jpg,0,mulher,train\nhard_180_0011.jpg,1,mulher,train\nmid_410_1111.jpg,1,mulher,train\nmid_22_1101.jpg,1,homem,train\nreal_00791.jpg,0,homem,train\nmid_9_1110.jpg,1,homem,train\neasy_114_1110.jpg,1,homem,train\neasy_69_1011.jpg,1,mulher,train\nmid_429_1110.jpg,1,mulher,train\nreal_00669.jpg,0,mulher,train\nhard_19_1111.jpg,1,mulher,train\nmid_346_0011.jpg,1,mulher,train\neasy_175_0001.jpg,1,mulher,train\nreal_01050.jpg,0,mulher,train\nreal_00850.jpg,0,mulher,train\nreal_00411.jpg,0,homem,train\nreal_00876.jpg,0,homem,train\nmid_224_1111.jpg,1,mulher,train\nreal_01054.jpg,0,mulher,train\nmid_480_1111.jpg,1,homem,train\nreal_00310.jpg,0,mulher,train\nreal_00289.jpg,0,mulher,train\nreal_00303.jpg,0,mulher,train\nreal_00228.jpg,0,mulher,train\nreal_00440.jpg,0,homem,train\neasy_200_1110.jpg,1,homem,train\nreal_00799.jpg,0,mulher,train\nreal_00554.jpg,0,homem,train\nmid_409_1101.jpg,1,mulher,train\nreal_00864.jpg,0,mulher,train\nreal_00716.jpg,0,homem,train\nmid_459_1001.jpg,1,mulher,train\nmid_49_1111.jpg,1,homem,train\nmid_95_1100.jpg,1,homem,train\nmid_254_1111.jpg,1,homem,train\nreal_00297.jpg,0,mulher,train\nreal_00393.jpg,0,mulher,train\nreal_00964.jpg,0,mulher,train\nreal_00153.jpg,0,homem,train\nreal_00066.jpg,0,mulher,train\nreal_00503.jpg,0,homem,train\nreal_00135.jpg,0,mulher,train\nreal_00148.jpg,0,mulher,train\nreal_00223.jpg,0,mulher,train\nhard_212_1101.jpg,1,homem,train\nreal_00715.jpg,0,mulher,train\nreal_00948.jpg,0,homem,train\nmid_437_1101.jpg,1,mulher,train\nreal_00624.jpg,0,homem,train\nmid_42_1111.jpg,1,homem,train\nmid_19_1000.jpg,1,homem,train\nhard_88_1111.jpg,1,homem,train\nhard_141_0100.jpg,1,mulher,train\nreal_00156.jpg,0,homem,train\nreal_00782.jpg,0,mulher,train\nreal_00003.jpg,0,mulher,train\nreal_00328.jpg,0,mulher,train\nreal_00262.jpg,0,homem,train\nmid_394_1111.jpg,1,homem,train\nhard_124_1100.jpg,1,homem,train\nreal_00318.jpg,0,mulher,train\nreal_00930.jpg,0,homem,train\nreal_00298.jpg,0,mulher,train\nmid_441_0011.jpg,1,homem,train\nmid_109_1111.jpg,1,mulher,train\nmid_27_1111.jpg,1,mulher,train\nreal_00619.jpg,0,homem,train\neasy_225_0010.jpg,1,mulher,train\nhard_70_1100.jpg,1,mulher,train\nmid_246_0011.jpg,1,mulher,train\neasy_60_1111.jpg,1,homem,train\nreal_00788.jpg,0,mulher,train\nmid_5_0011.jpg,1,homem,train\nreal_00831.jpg,0,mulher,train\nreal_00798.jpg,0,homem,train\nhard_202_1101.jpg,1,mulher,train\nreal_00965.jpg,0,mulher,train\neasy_195_1110.jpg,1,mulher,train\nmid_200_1111.jpg,1,mulher,train\neasy_141_0111.jpg,1,mulher,train\nreal_00631.jpg,0,mulher,train\nreal_00435.jpg,0,homem,train\nreal_00553.jpg,0,mulher,train\nreal_00544.jpg,0,mulher,train\nmid_341_1110.jpg,1,homem,train\nreal_00642.jpg,0,mulher,train\neasy_167_0110.jpg,1,mulher,train\nmid_155_1100.jpg,1,mulher,train\nreal_00281.jpg,0,mulher,train\nreal_00922.jpg,0,homem,train\nreal_00405.jpg,0,mulher,train\nreal_00380.jpg,0,homem,train\nreal_00709.jpg,0,mulher,train\nmid_403_1111.jpg,1,mulher,train\nmid_252_1101.jpg,1,homem,train\neasy_79_0001.jpg,1,homem,train\nreal_00886.jpg,0,homem,train\nreal_00425.jpg,0,mulher,train\nreal_00013.jpg,0,homem,train\nreal_00291.jpg,0,mulher,train\nreal_00494.jpg,0,mulher,train\nmid_100_1110.jpg,1,homem,train\nmid_372_0011.jpg,1,mulher,train\nmid_253_0101.jpg,1,mulher,train\nmid_449_1111.jpg,1,homem,train\nreal_00445.jpg,0,homem,train\nreal_00504.jpg,0,homem,train\nreal_01010.jpg,0,mulher,train\nreal_00023.jpg,0,mulher,train\nmid_37_1111.jpg,1,mulher,train\nreal_00539.jpg,0,mulher,train\nhard_14_1111.jpg,1,mulher,train\nreal_01071.jpg,0,homem,train\nreal_00976.jpg,0,mulher,train\nreal_00346.jpg,0,mulher,train\nreal_00350.jpg,0,mulher,train\nreal_00575.jpg,0,homem,train\nmid_233_1111.jpg,1,homem,train\nreal_00224.jpg,0,homem,train\neasy_139_0110.jpg,1,homem,train\nreal_01045.jpg,0,mulher,train\neasy_16_1111.jpg,1,mulher,train\nreal_00572.jpg,0,homem,train\nmid_388_1111.jpg,1,homem,train\nmid_304_1111.jpg,1,mulher,train\nreal_00580.jpg,0,homem,train\nmid_89_1111.jpg,1,homem,train\neasy_206_0011.jpg,1,mulher,train\nmid_93_1111.jpg,1,homem,train\neasy_106_0011.jpg,1,mulher,train\nreal_00128.jpg,0,mulher,train\nreal_00648.jpg,0,homem,train\nmid_71_1111.jpg,1,mulher,train\nreal_00144.jpg,0,mulher,train\nmid_462_1101.jpg,1,mulher,train\nhard_210_1100.jpg,1,homem,train\nmid_213_1111.jpg,1,homem,train\nreal_00638.jpg,0,homem,train\nreal_00695.jpg,0,homem,train\nreal_00758.jpg,0,mulher,train\nreal_00661.jpg,0,mulher,train\nreal_00200.jpg,0,homem,train\nreal_00266.jpg,0,mulher,train\nreal_00032.jpg,0,homem,train\nreal_00277.jpg,0,mulher,train\nhard_240_1101.jpg,1,homem,train\nreal_00558.jpg,0,homem,train\nreal_00035.jpg,0,homem,train\nreal_00432.jpg,0,mulher,train\nreal_00529.jpg,0,mulher,train\neasy_235_1111.jpg,1,mulher,train\nreal_00932.jpg,0,homem,train\neasy_12_1110.jpg,1,mulher,train\nmid_55_1111.jpg,1,mulher,train\nreal_00606.jpg,0,homem,train\nhard_116_1101.jpg,1,mulher,train\nmid_64_1100.jpg,1,mulher,train\nreal_00513.jpg,0,mulher,train\neasy_152_0011.jpg,1,homem,train\neasy_113_0011.jpg,1,homem,train\neasy_237_0011.jpg,1,mulher,train\nreal_00842.jpg,0,homem,train\nreal_00929.jpg,0,mulher,train\nmid_287_0001.jpg,1,mulher,train\nreal_00082.jpg,0,homem,train\neasy_201_0001.jpg,1,mulher,train\nhard_9_1111.jpg,1,mulher,train\nhard_95_0001.jpg,1,mulher,train\nmid_232_1111.jpg,1,mulher,train\nmid_298_0001.jpg,1,homem,train\nhard_96_1110.jpg,1,mulher,train\nreal_00487.jpg,0,mulher,train\nreal_00427.jpg,0,homem,train\nreal_00902.jpg,0,homem,train\nreal_00792.jpg,0,mulher,train\nmid_217_1111.jpg,1,mulher,train\nreal_00655.jpg,0,homem,train\nreal_00274.jpg,0,mulher,train\neasy_189_1101.jpg,1,homem,train\nreal_00838.jpg,0,mulher,train\neasy_204_0101.jpg,1,mulher,train\nreal_00420.jpg,0,homem,train\nreal_00653.jpg,0,homem,train\nreal_00516.jpg,0,mulher,train\nmid_12_1111.jpg,1,mulher,train\nreal_00710.jpg,0,homem,train\nhard_216_0111.jpg,1,homem,train\nreal_00563.jpg,0,homem,train\neasy_49_1000.jpg,1,homem,train\nreal_00217.jpg,0,homem,train\neasy_82_0001.jpg,1,mulher,train\nmid_146_1111.jpg,1,mulher,train\nreal_00163.jpg,0,homem,train\nreal_00934.jpg,0,mulher,train\nmid_85_0011.jpg,1,homem,train\nmid_300_0111.jpg,1,mulher,train\neasy_184_0110.jpg,1,homem,train\nreal_00381.jpg,0,mulher,train\nhard_147_1111.jpg,1,homem,train\nreal_00874.jpg,0,mulher,train\nreal_00823.jpg,0,homem,train\nreal_00833.jpg,0,mulher,train\neasy_76_1111.jpg,1,mulher,train\nreal_00742.jpg,0,homem,train\nreal_00870.jpg,0,homem,train\nreal_00866.jpg,0,mulher,train\nmid_455_0101.jpg,1,mulher,train\nreal_00467.jpg,0,homem,train\nmid_227_1111.jpg,1,mulher,train\nmid_379_1100.jpg,1,homem,train\nreal_00940.jpg,0,homem,train\nreal_00242.jpg,0,mulher,train\nreal_00527.jpg,0,mulher,train\nreal_00942.jpg,0,homem,train\nreal_01048.jpg,0,mulher,train\nmid_214_1011.jpg,1,mulher,train\nreal_00388.jpg,0,homem,train\nreal_00387.jpg,0,homem,train\nhard_44_1111.jpg,1,mulher,train\nreal_00231.jpg,0,homem,train\nreal_00759.jpg,0,homem,train\nmid_412_0011.jpg,1,mulher,train\nreal_00413.jpg,0,homem,train\nreal_00279.jpg,0,mulher,train\nreal_00789.jpg,0,homem,train\nreal_00679.jpg,0,mulher,train\neasy_218_0011.jpg,1,mulher,train\neasy_163_1001.jpg,1,homem,train\nmid_230_1110.jpg,1,mulher,train\neasy_107_1011.jpg,1,mulher,train\nreal_00481.jpg,0,mulher,train\nreal_00531.jpg,0,mulher,train\nhard_173_1110.jpg,1,homem,train\neasy_97_1111.jpg,1,homem,train\nmid_291_1100.jpg,1,homem,train\nreal_01056.jpg,0,mulher,train\nreal_00026.jpg,0,mulher,train\nreal_00473.jpg,0,mulher,train\nmid_357_0011.jpg,1,mulher,train\nreal_01027.jpg,0,mulher,train\nreal_00286.jpg,0,mulher,train\nreal_01049.jpg,0,mulher,train\nmid_425_1111.jpg,1,homem,train\nreal_00997.jpg,0,homem,train\nhard_104_1001.jpg,1,mulher,train\nreal_00202.jpg,0,homem,train\nhard_54_0001.jpg,1,homem,train\neasy_135_0011.jpg,1,mulher,train\nreal_00924.jpg,0,mulher,train\neasy_26_1100.jpg,1,mulher,train\nreal_00232.jpg,0,homem,train\neasy_238_1111.jpg,1,homem,train\nmid_102_1100.jpg,1,homem,train\nreal_00021.jpg,0,homem,train\nreal_00794.jpg,0,mulher,train\nmid_187_1101.jpg,1,mulher,train\nreal_00282.jpg,0,mulher,train\nhard_178_1001.jpg,1,homem,train\nreal_00836.jpg,0,homem,train\nreal_00315.jpg,0,mulher,train\neasy_83_1110.jpg,1,homem,train\nreal_00900.jpg,0,mulher,train\nmid_131_1111.jpg,1,homem,train\nreal_00680.jpg,0,homem,train\nreal_00322.jpg,0,mulher,train\nreal_00536.jpg,0,homem,train\nmid_293_1101.jpg,1,mulher,train\nhard_149_0011.jpg,1,mulher,train\nreal_00699.jpg,0,mulher,train\nreal_00694.jpg,0,homem,train\nhard_98_1101.jpg,1,mulher,train\nreal_00199.jpg,0,homem,train\nmid_236_1111.jpg,1,homem,train\nreal_01013.jpg,0,mulher,train\nhard_31_0011.jpg,1,homem,train\nreal_00423.jpg,0,homem,train\nreal_00160.jpg,0,mulher,train\nreal_00621.jpg,0,homem,train\nreal_00488.jpg,0,homem,train\nreal_00444.jpg,0,mulher,train\nmid_116_1111.jpg,1,homem,train\nhard_27_1111.jpg,1,mulher,train\nreal_00909.jpg,0,mulher,train\nreal_00971.jpg,0,homem,train\nreal_00263.jpg,0,mulher,train\nhard_126_0111.jpg,1,homem,train\nreal_00583.jpg,0,homem,train\nreal_00950.jpg,0,homem,train\nhard_190_1111.jpg,1,mulher,train\nreal_00326.jpg,0,mulher,train\nmid_370_1111.jpg,1,homem,train\nmid_191_0111.jpg,1,mulher,train\nmid_381_1101.jpg,1,mulher,train\nreal_00100.jpg,0,mulher,train\nreal_00378.jpg,0,mulher,train\nmid_168_1111.jpg,1,mulher,train\nmid_57_1111.jpg,1,homem,train\neasy_214_0011.jpg,1,homem,train\nreal_00750.jpg,0,mulher,train\neasy_53_0001.jpg,1,homem,train\nreal_00545.jpg,0,mulher,train\nreal_00797.jpg,0,homem,train\nreal_00276.jpg,0,mulher,train\nreal_00344.jpg,0,mulher,train\nreal_00141.jpg,0,mulher,train\neasy_146_1110.jpg,1,mulher,train\nmid_382_0011.jpg,1,homem,train\neasy_174_1110.jpg,1,homem,train\nmid_13_1111.jpg,1,mulher,train\neasy_30_1001.jpg,1,mulher,train\nreal_00927.jpg,0,homem,train\nreal_00780.jpg,0,mulher,train\neasy_149_1001.jpg,1,mulher,train\nreal_00449.jpg,0,mulher,train\nmid_24_1111.jpg,1,homem,train\nreal_00428.jpg,0,homem,train\nreal_00226.jpg,0,mulher,train\neasy_37_0010.jpg,1,homem,train\nreal_00520.jpg,0,mulher,train\nreal_00947.jpg,0,homem,train\nreal_00364.jpg,0,mulher,train\nmid_270_0101.jpg,1,homem,train\nhard_81_1101.jpg,1,mulher,train\nreal_00351.jpg,0,mulher,train\nhard_109_1100.jpg,1,mulher,train\nmid_210_0011.jpg,1,homem,train\nreal_00285.jpg,0,mulher,train\nreal_00537.jpg,0,mulher,train\nreal_00713.jpg,0,homem,train\nmid_407_1111.jpg,1,mulher,train\nreal_00238.jpg,0,homem,train\nreal_00828.jpg,0,mulher,train\nmid_424_1111.jpg,1,mulher,train\nhard_33_1011.jpg,1,homem,train\nhard_67_0001.jpg,1,mulher,train\nreal_00176.jpg,0,mulher,train\nreal_00800.jpg,0,mulher,train\nreal_00490.jpg,0,homem,train\nmid_283_0111.jpg,1,homem,train\nmid_302_1100.jpg,1,homem,train\nmid_154_1111.jpg,1,homem,train\nreal_00134.jpg,0,mulher,train\nreal_00854.jpg,0,homem,train\nreal_00007.jpg,0,mulher,train\neasy_228_0011.jpg,1,mulher,train\nreal_00956.jpg,0,homem,train\nreal_00366.jpg,0,mulher,train\nreal_00665.jpg,0,mulher,train\nreal_00667.jpg,0,homem,train\nreal_00203.jpg,0,homem,train\nreal_00339.jpg,0,mulher,train\nreal_00416.jpg,0,homem,train\nhard_46_0011.jpg,1,mulher,train\nreal_00301.jpg,0,mulher,train\nreal_01043.jpg,0,mulher,train\nmid_312_1111.jpg,1,homem,train\nhard_200_1111.jpg,1,homem,train\neasy_131_1001.jpg,1,mulher,train\nreal_00041.jpg,0,homem,train\neasy_239_1110.jpg,1,homem,train\nmid_284_1110.jpg,1,mulher,train\nmid_18_1111.jpg,1,homem,train\nmid_262_1111.jpg,1,mulher,train\nreal_00685.jpg,0,homem,train\nmid_268_0010.jpg,1,mulher,train\nmid_289_1101.jpg,1,homem,train\nmid_415_1111.jpg,1,mulher,train\neasy_165_1010.jpg,1,homem,train\nreal_00996.jpg,0,mulher,train\nreal_00472.jpg,0,homem,train\nreal_00684.jpg,0,homem,train\nreal_00636.jpg,0,mulher,train\nhard_205_0011.jpg,1,homem,train\nreal_00475.jpg,0,mulher,train\nmid_135_1110.jpg,1,mulher,train\nmid_205_0011.jpg,1,mulher,train\nreal_00561.jpg,0,homem,train\nmid_46_1111.jpg,1,homem,train\nreal_00574.jpg,0,homem,train\nreal_00592.jpg,0,mulher,train\nreal_00779.jpg,0,mulher,train\nreal_00253.jpg,0,homem,train\nreal_00641.jpg,0,mulher,train\nreal_00770.jpg,0,mulher,train\nreal_00670.jpg,0,homem,train\nhard_238_1000.jpg,1,homem,train\nreal_00396.jpg,0,homem,train\nreal_00034.jpg,0,homem,train\nmid_290_1100.jpg,1,homem,train\nmid_211_1100.jpg,1,homem,train\neasy_127_0001.jpg,1,homem,train\nreal_00602.jpg,0,homem,train\nreal_00025.jpg,0,mulher,train\nmid_123_1111.jpg,1,mulher,train\nreal_01061.jpg,0,mulher,train\nmid_297_0010.jpg,1,mulher,train\nreal_00972.jpg,0,mulher,train\nmid_206_1111.jpg,1,mulher,train\nreal_00448.jpg,0,mulher,train\nreal_00088.jpg,0,mulher,train\nreal_00522.jpg,0,homem,train\nhard_153_1011.jpg,1,mulher,train\nreal_00700.jpg,0,mulher,train\nmid_276_0011.jpg,1,homem,train\nreal_00149.jpg,0,homem,train\nhard_103_1101.jpg,1,mulher,train\neasy_34_1100.jpg,1,mulher,train\nreal_00635.jpg,0,mulher,train\nreal_00207.jpg,0,mulher,train\nreal_00650.jpg,0,homem,train\nmid_45_0010.jpg,1,homem,train\nreal_00192.jpg,0,mulher,train\nmid_74_1111.jpg,1,mulher,train\nhard_23_1110.jpg,1,mulher,train\nreal_00718.jpg,0,homem,train\nreal_00743.jpg,0,mulher,train\nmid_126_1110.jpg,1,mulher,train\nhard_192_1111.jpg,1,mulher,train\nreal_00348.jpg,0,mulher,train\nreal_00969.jpg,0,mulher,train\nmid_401_0111.jpg,1,mulher,train\nreal_00591.jpg,0,mulher,train\nreal_00657.jpg,0,homem,train\nreal_00317.jpg,0,mulher,train\nreal_01005.jpg,0,homem,train\nmid_72_1111.jpg,1,mulher,train\neasy_104_1000.jpg,1,homem,train\nreal_00152.jpg,0,homem,train\nhard_136_1101.jpg,1,mulher,train\nreal_00219.jpg,0,homem,train\nreal_00327.jpg,0,mulher,train\nreal_01039.jpg,0,mulher,train\nreal_00426.jpg,0,homem,train\nreal_00076.jpg,0,mulher,train\nreal_00314.jpg,0,mulher,train\nhard_121_1100.jpg,1,homem,train\neasy_14_1111.jpg,1,homem,train\nmid_83_1111.jpg,1,mulher,train\nreal_00243.jpg,0,mulher,train\nmid_342_1110.jpg,1,homem,train\nmid_404_0011.jpg,1,mulher,train\nreal_00704.jpg,0,homem,train\nreal_00857.jpg,0,mulher,train\nmid_359_1101.jpg,1,mulher,train\nreal_00233.jpg,0,homem,train\neasy_150_0011.jpg,1,mulher,train\nreal_00772.jpg,0,mulher,train\nmid_436_1111.jpg,1,homem,train\nreal_00244.jpg,0,mulher,train\nreal_00916.jpg,0,mulher,train\nreal_00070.jpg,0,homem,train\neasy_10_0001.jpg,1,homem,train\nreal_00183.jpg,0,mulher,train\nmid_277_1100.jpg,1,homem,train\nreal_00222.jpg,0,mulher,train\nreal_00644.jpg,0,homem,train\neasy_126_0111.jpg,1,homem,train\nreal_00430.jpg,0,homem,train\nhard_119_1100.jpg,1,mulher,train\neasy_215_1111.jpg,1,mulher,train\nmid_193_1111.jpg,1,homem,train\nhard_101_0011.jpg,1,mulher,train\nmid_212_1111.jpg,1,homem,train\nreal_00776.jpg,0,mulher,train\neasy_199_1100.jpg,1,mulher,train\nreal_00334.jpg,0,mulher,train\nhard_30_1101.jpg,1,mulher,train\nreal_00353.jpg,0,mulher,train\nmid_51_1111.jpg,1,homem,train\nmid_143_1100.jpg,1,mulher,train\nhard_168_0011.jpg,1,homem,train\nreal_00796.jpg,0,homem,train\nreal_00960.jpg,0,mulher,train\nreal_00091.jpg,0,homem,train\nreal_01012.jpg,0,mulher,train\nreal_00730.jpg,0,homem,train\nhard_7_1111.jpg,1,mulher,train\nreal_00514.jpg,0,mulher,train\nreal_01025.jpg,0,mulher,train\nreal_01034.jpg,0,mulher,train\nreal_01016.jpg,0,mulher,train\nmid_320_1111.jpg,1,homem,train\nmid_469_1111.jpg,1,mulher,train\nreal_00206.jpg,0,homem,train\nreal_00959.jpg,0,homem,train\nreal_00479.jpg,0,mulher,train\nmid_192_1111.jpg,1,mulher,train\neasy_186_1001.jpg,1,mulher,train\nreal_00824.jpg,0,mulher,train\nreal_00967.jpg,0,mulher,train\nmid_311_0011.jpg,1,homem,train\nreal_00820.jpg,0,mulher,train\nmid_157_1111.jpg,1,homem,train\neasy_43_1100.jpg,1,mulher,train\neasy_45_1010.jpg,1,mulher,train\nmid_265_1111.jpg,1,mulher,train\neasy_102_0101.jpg,1,mulher,train\nmid_399_0011.jpg,1,mulher,train\nreal_00058.jpg,0,mulher,train\nmid_38_1111.jpg,1,mulher,train\nreal_01040.jpg,0,mulher,train\nhard_117_1110.jpg,1,mulher,train\neasy_138_0101.jpg,1,homem,train\nreal_00015.jpg,0,homem,train\nreal_00894.jpg,0,homem,train\nhard_125_1100.jpg,1,homem,train\nreal_00505.jpg,0,homem,train\neasy_153_1110.jpg,1,mulher,train\nreal_00386.jpg,0,mulher,train\nhard_49_0010.jpg,1,homem,train\nreal_00129.jpg,0,homem,train\nreal_00126.jpg,0,mulher,train\nreal_00891.jpg,0,homem,train\nreal_00391.jpg,0,homem,train\nreal_00628.jpg,0,mulher,train\nreal_00540.jpg,0,homem,train\nreal_00419.jpg,0,homem,train\nreal_00404.jpg,0,homem,train\nreal_00270.jpg,0,mulher,train\nmid_397_1111.jpg,1,mulher,train\nreal_00495.jpg,0,mulher,train\nhard_16_1111.jpg,1,mulher,train\nmid_363_1101.jpg,1,mulher,train\nhard_38_1110.jpg,1,homem,train\neasy_202_0011.jpg,1,homem,train\nmid_430_1111.jpg,1,homem,train\nmid_96_1111.jpg,1,homem,train\nreal_00790.jpg,0,mulher,train\nreal_00477.jpg,0,mulher,train\nreal_00826.jpg,0,mulher,train\neasy_1_1110.jpg,1,homem,train\nreal_00439.jpg,0,mulher,train\nreal_00080.jpg,0,homem,train\neasy_130_1101.jpg,1,homem,train\nreal_00269.jpg,0,mulher,train\nreal_00045.jpg,0,homem,train\nreal_00577.jpg,0,homem,train\nmid_476_1111.jpg,1,homem,train\nmid_445_1111.jpg,1,mulher,train\nhard_42_1111.jpg,1,homem,train\nreal_00807.jpg,0,homem,train\nreal_00022.jpg,0,homem,train\nhard_171_1101.jpg,1,homem,train\neasy_54_1010.jpg,1,mulher,train\neasy_70_0011.jpg,1,mulher,train\neasy_176_1000.jpg,1,mulher,train\nreal_00394.jpg,0,homem,train\nhard_62_1010.jpg,1,homem,train\nmid_28_1111.jpg,1,homem,train\nmid_457_1100.jpg,1,mulher,train\nhard_107_1111.jpg,1,mulher,train\neasy_217_1111.jpg,1,homem,train\nhard_181_0001.jpg,1,mulher,train\nreal_00132.jpg,0,mulher,train\nreal_00899.jpg,0,mulher,train\neasy_29_0011.jpg,1,homem,train\nreal_00521.jpg,0,mulher,train\nreal_00168.jpg,0,homem,train\nreal_00955.jpg,0,mulher,train\nreal_00497.jpg,0,mulher,train\nmid_199_0011.jpg,1,mulher,train\nmid_34_1111.jpg,1,mulher,train\neasy_230_1010.jpg,1,homem,train\nreal_00705.jpg,0,homem,train\nreal_00884.jpg,0,homem,train\neasy_36_1111.jpg,1,homem,train\nreal_00579.jpg,0,homem,train\nreal_00827.jpg,0,homem,train\nhard_8_1000.jpg,1,homem,train\nreal_00746.jpg,0,mulher,train\nhard_66_1111.jpg,1,mulher,train\nmid_223_1111.jpg,1,mulher,train\nreal_00733.jpg,0,homem,train\nreal_00173.jpg,0,homem,train\nreal_01018.jpg,0,mulher,train\nmid_255_0011.jpg,1,mulher,train\nmid_144_1110.jpg,1,homem,train\nmid_238_1111.jpg,1,mulher,train\nhard_172_1100.jpg,1,homem,train\nreal_00735.jpg,0,homem,train\nreal_00953.jpg,0,mulher,train\nhard_76_1100.jpg,1,homem,train\neasy_22_1010.jpg,1,mulher,train\nreal_00469.jpg,0,homem,train\nreal_00363.jpg,0,mulher,train\nreal_00368.jpg,0,mulher,train\nreal_00205.jpg,0,mulher,train\nreal_00001.jpg,0,homem,train\nreal_00547.jpg,0,mulher,train\nreal_00125.jpg,0,mulher,train\nreal_00053.jpg,0,mulher,train\neasy_23_1100.jpg,1,mulher,train\nreal_00728.jpg,0,homem,train\nreal_00608.jpg,0,homem,train\nhard_184_0011.jpg,1,mulher,train\nreal_01069.jpg,0,mulher,train\nreal_00570.jpg,0,homem,train\nreal_00740.jpg,0,mulher,train\nreal_00117.jpg,0,homem,train\nhard_132_0111.jpg,1,mulher,train\nreal_00612.jpg,0,mulher,train\nhard_57_0110.jpg,1,mulher,train\nreal_00138.jpg,0,homem,train\nreal_00564.jpg,0,homem,train\nreal_00036.jpg,0,homem,train\nreal_00295.jpg,0,mulher,train\nreal_00943.jpg,0,homem,train\neasy_32_1100.jpg,1,homem,train\nmid_112_0001.jpg,1,mulher,train\neasy_211_1100.jpg,1,mulher,train\nreal_00379.jpg,0,mulher,train\nhard_165_1000.jpg,1,mulher,train\neasy_121_0011.jpg,1,homem,train\nreal_00601.jpg,0,homem,train\neasy_39_0011.jpg,1,homem,train\nreal_00923.jpg,0,homem,train\neasy_98_0001.jpg,1,homem,train\nmid_137_1011.jpg,1,homem,train\neasy_28_1100.jpg,1,mulher,train\neasy_231_1011.jpg,1,mulher,train\neasy_229_1000.jpg,1,mulher,train\nreal_00484.jpg,0,mulher,train\nreal_00732.jpg,0,homem,train\nreal_00808.jpg,0,mulher,train\nreal_00047.jpg,0,homem,train\nreal_00218.jpg,0,homem,train\nreal_01074.jpg,0,homem,train\nmid_473_0011.jpg,1,mulher,train\nreal_00562.jpg,0,homem,train\nreal_00255.jpg,0,mulher,train\nreal_00170.jpg,0,homem,train\nmid_271_1111.jpg,1,homem,train\nreal_00020.jpg,0,homem,train\nreal_00489.jpg,0,mulher,train\nreal_01079.jpg,0,homem,train\nreal_01058.jpg,0,homem,train\neasy_9_1010.jpg,1,mulher,train\nmid_245_1100.jpg,1,mulher,train\nmid_319_1111.jpg,1,mulher,train\nreal_00875.jpg,0,mulher,train\nmid_417_1111.jpg,1,mulher,train\nreal_00019.jpg,0,mulher,train\nreal_00920.jpg,0,mulher,train\nreal_00869.jpg,0,mulher,train\nreal_00260.jpg,0,homem,train\nreal_00744.jpg,0,mulher,train\nreal_00139.jpg,0,homem,train\nreal_00673.jpg,0,homem,train\nhard_112_1111.jpg,1,mulher,train\nmid_44_1111.jpg,1,homem,train\nreal_00830.jpg,0,mulher,train\nreal_00458.jpg,0,mulher,train\nreal_00225.jpg,0,homem,train\nmid_257_1111.jpg,1,mulher,train\nhard_85_1111.jpg,1,mulher,train\neasy_19_0111.jpg,1,mulher,train\nreal_00937.jpg,0,mulher,train\nmid_421_0011.jpg,1,homem,train\nreal_00871.jpg,0,mulher,train\nhard_179_1111.jpg,1,mulher,train\nmid_318_1111.jpg,1,homem,train\nmid_374_0011.jpg,1,homem,train\nreal_00006.jpg,0,mulher,train\nhard_56_0100.jpg,1,mulher,train\nreal_00040.jpg,0,homem,train\neasy_156_0010.jpg,1,mulher,train\nmid_266_1101.jpg,1,mulher,train\nreal_00158.jpg,0,homem,train\nreal_00187.jpg,0,homem,train\nhard_221_0101.jpg,1,mulher,train\nreal_00069.jpg,0,homem,train\nmid_454_1111.jpg,1,mulher,train\nreal_00622.jpg,0,mulher,train\nmid_82_1111.jpg,1,mulher,train\nmid_167_1100.jpg,1,mulher,train\nhard_51_1111.jpg,1,mulher,train\nmid_77_1110.jpg,1,homem,train\nreal_00438.jpg,0,mulher,train\nhard_84_0011.jpg,1,mulher,train\neasy_194_1011.jpg,1,mulher,train\nmid_474_1110.jpg,1,mulher,train\nreal_00043.jpg,0,mulher,train\nreal_00645.jpg,0,homem,train\nmid_282_1100.jpg,1,homem,train\nmid_467_1111.jpg,1,mulher,train\nreal_00130.jpg,0,homem,train\neasy_207_1000.jpg,1,mulher,train\nreal_00845.jpg,0,homem,train\nreal_00442.jpg,0,mulher,train\nreal_00033.jpg,0,homem,train\neasy_81_1111.jpg,1,mulher,train\nmid_153_1111.jpg,1,homem,train\nreal_00568.jpg,0,homem,train\nreal_01055.jpg,0,mulher,train\nmid_235_1100.jpg,1,homem,train\nhard_130_1111.jpg,1,homem,train\nreal_00422.jpg,0,homem,train\nhard_193_1111.jpg,1,mulher,train\nreal_00122.jpg,0,mulher,train\nhard_79_1110.jpg,1,homem,train\nreal_00998.jpg,0,mulher,train\nhard_234_1100.jpg,1,mulher,train\neasy_2_1111.jpg,1,mulher,train\neasy_62_0001.jpg,1,homem,train\nmid_156_1111.jpg,1,mulher,train\nhard_92_1001.jpg,1,mulher,train\nmid_460_1111.jpg,1,mulher,train\nreal_00236.jpg,0,mulher,train\neasy_196_0010.jpg,1,homem,train\nreal_01003.jpg,0,homem,train\nmid_15_1111.jpg,1,mulher,train\nreal_00585.jpg,0,mulher,train\nreal_00613.jpg,0,mulher,train\nreal_00904.jpg,0,homem,train\nmid_88_1110.jpg,1,homem,train\nmid_413_1111.jpg,1,homem,train\nhard_43_1111.jpg,1,mulher,train\nmid_368_0110.jpg,1,mulher,train\nreal_00320.jpg,0,mulher,train\nmid_105_1101.jpg,1,mulher,train\nreal_01053.jpg,0,mulher,train\nmid_471_1111.jpg,1,homem,train\nreal_01052.jpg,0,mulher,train\nhard_188_0100.jpg,1,mulher,train\nreal_00048.jpg,0,mulher,train\neasy_147_1110.jpg,1,mulher,train\nreal_01057.jpg,0,mulher,train\nreal_00958.jpg,0,mulher,train\nmid_138_0011.jpg,1,mulher,train\neasy_96_0011.jpg,1,mulher,train\nreal_00630.jpg,0,mulher,train\nmid_324_1111.jpg,1,homem,train\nhard_236_1001.jpg,1,mulher,train\nreal_00249.jpg,0,homem,train\nmid_234_0010.jpg,1,mulher,train\nreal_00409.jpg,0,homem,train\nreal_01067.jpg,0,homem,train\nreal_00625.jpg,0,mulher,train\neasy_6_1110.jpg,1,mulher,train\nmid_21_0011.jpg,1,homem,train\nmid_294_1110.jpg,1,mulher,train\nhard_52_1111.jpg,1,homem,train\nhard_237_0010.jpg,1,mulher,train\neasy_78_1111.jpg,1,homem,train\nmid_101_1111.jpg,1,homem,train\nreal_00300.jpg,0,mulher,train\neasy_145_1000.jpg,1,mulher,train\nreal_00755.jpg,0,mulher,train\nmid_351_0110.jpg,1,mulher,train\nreal_01064.jpg,0,mulher,train\nmid_30_1110.jpg,1,mulher,train\nhard_64_0010.jpg,1,mulher,train\nreal_00118.jpg,0,homem,train\nreal_00787.jpg,0,mulher,train\nreal_00005.jpg,0,homem,train\nreal_00360.jpg,0,mulher,train\nreal_00305.jpg,0,mulher,train\nreal_00417.jpg,0,mulher,train\nreal_00949.jpg,0,homem,train\nreal_00507.jpg,0,homem,train\nreal_00042.jpg,0,mulher,train\nreal_00059.jpg,0,mulher,train\nmid_26_1101.jpg,1,mulher,train\neasy_40_1111.jpg,1,homem,train\nreal_00898.jpg,0,homem,train\nreal_00174.jpg,0,mulher,train\nreal_00541.jpg,0,mulher,train\nreal_00113.jpg,0,mulher,train\nreal_00821.jpg,0,mulher,train\nreal_00989.jpg,0,mulher,train\nreal_00261.jpg,0,mulher,train\nreal_00853.jpg,0,homem,train\nmid_390_0011.jpg,1,mulher,train\nreal_01036.jpg,0,mulher,train\nreal_01081.jpg,0,mulher,train\nreal_00383.jpg,0,homem,train\nreal_00803.jpg,0,mulher,train\neasy_59_0010.jpg,1,homem,train\nmid_237_1111.jpg,1,homem,train\nhard_163_0011.jpg,1,mulher,train\nreal_00410.jpg,0,homem,train\nreal_00861.jpg,0,homem,train\nreal_01033.jpg,0,mulher,train\nhard_155_0010.jpg,1,mulher,train\nhard_219_1101.jpg,1,mulher,train\nmid_286_1100.jpg,1,mulher,train\nreal_00374.jpg,0,mulher,train\nreal_00659.jpg,0,mulher,train\nmid_161_0110.jpg,1,mulher,train\nreal_00483.jpg,0,mulher,train\nmid_315_1101.jpg,1,mulher,train\nmid_323_1101.jpg,1,homem,train\nhard_58_0001.jpg,1,homem,train\nmid_367_0010.jpg,1,mulher,train\nreal_00039.jpg,0,homem,train\nmid_204_1111.jpg,1,mulher,train\neasy_48_0011.jpg,1,homem,train\neasy_64_1111.jpg,1,mulher,train\nmid_73_1110.jpg,1,mulher,train\nhard_69_0011.jpg,1,homem,train\neasy_57_0010.jpg,1,mulher,train\nreal_00977.jpg,0,homem,train\nmid_218_1011.jpg,1,homem,train\nreal_00441.jpg,0,homem,train\nmid_4_1111.jpg,1,mulher,train\neasy_170_0010.jpg,1,mulher,train\nmid_243_0011.jpg,1,homem,train\nmid_383_1100.jpg,1,homem,train\nhard_191_0011.jpg,1,mulher,train\nmid_258_1110.jpg,1,mulher,train\nreal_00177.jpg,0,homem,train\nmid_215_0011.jpg,1,mulher,train\nmid_104_1100.jpg,1,mulher,train\nreal_00234.jpg,0,homem,train\nreal_00476.jpg,0,mulher,train\nreal_00104.jpg,0,homem,train\neasy_181_1110.jpg,1,mulher,train\nreal_00868.jpg,0,mulher,train\nreal_00518.jpg,0,homem,train\nreal_00852.jpg,0,homem,train\nreal_00247.jpg,0,homem,train\nreal_00681.jpg,0,mulher,train\nhard_206_0001.jpg,1,mulher,train\neasy_198_0011.jpg,1,mulher,train\neasy_124_1110.jpg,1,mulher,train\neasy_120_0011.jpg,1,mulher,train\neasy_87_0110.jpg,1,mulher,train\nmid_181_0011.jpg,1,mulher,train\nreal_00429.jpg,0,mulher,train\nreal_00738.jpg,0,mulher,train\nmid_375_1110.jpg,1,mulher,train\nmid_201_1111.jpg,1,mulher,train\nreal_00805.jpg,0,homem,train\nreal_00382.jpg,0,homem,train\nreal_00336.jpg,0,mulher,train\nreal_00914.jpg,0,mulher,train\nmid_478_1101.jpg,1,mulher,train\nreal_00220.jpg,0,mulher,train\nhard_229_0100.jpg,1,homem,train\nreal_00447.jpg,0,mulher,train\neasy_56_1110.jpg,1,mulher,train\neasy_65_1111.jpg,1,mulher,train\nreal_00693.jpg,0,homem,train\nreal_00213.jpg,0,mulher,train\nreal_00137.jpg,0,mulher,train\nmid_264_1101.jpg,1,mulher,train\nreal_00729.jpg,0,homem,train\nmid_464_1111.jpg,1,mulher,train\nreal_00804.jpg,0,homem,train\nmid_8_1111.jpg,1,homem,train\neasy_44_0011.jpg,1,mulher,train\neasy_154_1100.jpg,1,mulher,train\nreal_00672.jpg,0,mulher,train\nreal_00712.jpg,0,homem,train\nreal_00970.jpg,0,mulher,train\nreal_00739.jpg,0,mulher,train\nreal_00150.jpg,0,mulher,train\nreal_00757.jpg,0,mulher,train\nmid_438_0010.jpg,1,mulher,train\nmid_128_1110.jpg,1,homem,train\neasy_94_0001.jpg,1,mulher,train\nreal_00919.jpg,0,homem,train\nreal_01077.jpg,0,mulher,train\neasy_100_1111.jpg,1,mulher,train\nmid_183_1111.jpg,1,homem,train\nhard_197_1100.jpg,1,mulher,train\nreal_00338.jpg,0,mulher,train\nreal_00110.jpg,0,homem,train\nreal_00793.jpg,0,mulher,train\neasy_80_0001.jpg,1,mulher,train\nhard_50_1101.jpg,1,homem,train\nreal_00587.jpg,0,homem,train\nhard_41_1111.jpg,1,homem,train\nreal_00618.jpg,0,mulher,train\nhard_169_0111.jpg,1,homem,train\nreal_00450.jpg,0,mulher,train\nmid_326_1111.jpg,1,mulher,train\nmid_170_1111.jpg,1,mulher,train\nmid_452_1111.jpg,1,homem,train\nreal_00103.jpg,0,homem,train\nmid_110_1100.jpg,1,mulher,train\nreal_01062.jpg,0,homem,train\neasy_209_0001.jpg,1,homem,train\nreal_01024.jpg,0,mulher,train\nhard_196_1101.jpg,1,mulher,train\nreal_00763.jpg,0,mulher,train\nreal_00856.jpg,0,mulher,train\nreal_00361.jpg,0,mulher,train\nreal_00595.jpg,0,mulher,train\nmid_310_0111.jpg,1,homem,train\nmid_228_1111.jpg,1,homem,train\nmid_150_1111.jpg,1,mulher,train\nreal_00806.jpg,0,mulher,train\nreal_00189.jpg,0,mulher,train\nmid_97_1111.jpg,1,mulher,train\nreal_00549.jpg,0,homem,train\nhard_232_0001.jpg,1,mulher,train\nreal_00627.jpg,0,mulher,train\nreal_00245.jpg,0,mulher,train\nreal_00599.jpg,0,mulher,train\neasy_183_1111.jpg,1,homem,train\nmid_325_0011.jpg,1,homem,train\nreal_00355.jpg,0,mulher,train\nreal_00401.jpg,0,homem,train\nreal_00652.jpg,0,homem,train\nhard_182_0011.jpg,1,mulher,train\nmid_350_0100.jpg,1,mulher,train\nreal_00528.jpg,0,mulher,train\neasy_109_1111.jpg,1,mulher,train\neasy_86_0011.jpg,1,homem,train\nhard_89_1110.jpg,1,mulher,train\nreal_00983.jpg,0,mulher,train\nhard_82_0011.jpg,1,mulher,train\nmid_340_1111.jpg,1,mulher,train\nmid_463_0100.jpg,1,homem,train\neasy_38_0010.jpg,1,mulher,train\nmid_148_0001.jpg,1,homem,train\nmid_90_1111.jpg,1,mulher,train\nreal_00230.jpg,0,homem,train\neasy_111_1101.jpg,1,mulher,train\nhard_83_1111.jpg,1,mulher,train\nhard_157_1111.jpg,1,mulher,train\nmid_365_1111.jpg,1,homem,train\nreal_00399.jpg,0,homem,train\nreal_00878.jpg,0,homem,train\neasy_13_1010.jpg,1,mulher,train\nreal_00626.jpg,0,homem,train\nmid_475_1111.jpg,1,mulher,train\nreal_00901.jpg,0,homem,train\nmid_52_1111.jpg,1,homem,train\nreal_00201.jpg,0,homem,train\nmid_249_1111.jpg,1,mulher,train\nreal_00677.jpg,0,mulher,train\nhard_167_0011.jpg,1,mulher,train\nmid_317_1101.jpg,1,mulher,train\neasy_112_1001.jpg,1,homem,train\neasy_8_0010.jpg,1,mulher,train\nhard_26_0011.jpg,1,mulher,train\nreal_00576.jpg,0,homem,train\nreal_00294.jpg,0,mulher,train\nreal_00214.jpg,0,mulher,train\nreal_00893.jpg,0,homem,train\neasy_7_1100.jpg,1,mulher,train\neasy_72_1111.jpg,1,mulher,train\nreal_00075.jpg,0,mulher,train\nreal_00640.jpg,0,homem,train\nmid_79_1111.jpg,1,mulher,train\neasy_95_0101.jpg,1,mulher,train\nhard_144_0011.jpg,1,mulher,train\nmid_222_1110.jpg,1,mulher,train\nreal_00676.jpg,0,homem,train\nmid_386_1110.jpg,1,mulher,train\nmid_229_1111.jpg,1,homem,train\nreal_00745.jpg,0,mulher,train\nmid_221_1100.jpg,1,mulher,train\nmid_80_1111.jpg,1,homem,train\neasy_35_0110.jpg,1,mulher,train\nreal_00844.jpg,0,mulher,train\nreal_00957.jpg,0,mulher,train\nreal_01070.jpg,0,mulher,train\nreal_00191.jpg,0,homem,train\nreal_00928.jpg,0,mulher,train\nhard_185_1100.jpg,1,mulher,train\nreal_00944.jpg,0,mulher,train\nreal_00912.jpg,0,mulher,train\nmid_54_1111.jpg,1,homem,train\nhard_161_1100.jpg,1,homem,train\nmid_177_1111.jpg,1,mulher,train\nmid_115_0011.jpg,1,homem,train\nreal_00892.jpg,0,mulher,train\nmid_124_1110.jpg,1,homem,train\nreal_00470.jpg,0,mulher,train\neasy_240_1110.jpg,1,mulher,train\neasy_178_1111.jpg,1,mulher,train\nreal_00848.jpg,0,homem,train\nreal_00646.jpg,0,homem,train\nmid_354_1110.jpg,1,mulher,train\nreal_01078.jpg,0,mulher,train\nmid_358_1100.jpg,1,mulher,train\nreal_01044.jpg,0,mulher,train\nreal_00196.jpg,0,mulher,train\nreal_00384.jpg,0,homem,train\nreal_00611.jpg,0,mulher,train\nmid_86_1111.jpg,1,homem,train\nreal_00446.jpg,0,mulher,train\nhard_37_1111.jpg,1,mulher,train\nmid_166_1100.jpg,1,homem,train\nreal_00062.jpg,0,homem,train\nreal_01014.jpg,0,mulher,train\nmid_338_1100.jpg,1,mulher,train\nreal_00347.jpg,0,mulher,train\nreal_00182.jpg,0,mulher,train\nreal_00979.jpg,0,homem,train\nreal_00543.jpg,0,homem,train\nmid_67_1111.jpg,1,homem,train\nreal_00883.jpg,0,homem,train\nreal_00781.jpg,0,mulher,train\nhard_94_1111.jpg,1,mulher,train\nreal_00012.jpg,0,homem,train\nreal_00501.jpg,0,mulher,train\nmid_165_0101.jpg,1,mulher,train\nreal_00323.jpg,0,mulher,train\nmid_92_1111.jpg,1,mulher,train\nmid_121_1110.jpg,1,mulher,train\nreal_01008.jpg,0,mulher,train\nmid_76_1111.jpg,1,mulher,train\nreal_00550.jpg,0,mulher,train\nreal_00918.jpg,0,homem,train\nreal_00717.jpg,0,mulher,train\nmid_479_1111.jpg,1,homem,train\nreal_00847.jpg,0,homem,train\nhard_110_1110.jpg,1,homem,train\nreal_00486.jpg,0,homem,train\nreal_00385.jpg,0,homem,train\nreal_00241.jpg,0,mulher,train\nreal_00802.jpg,0,mulher,train\nmid_98_1111.jpg,1,homem,train\nhard_187_0100.jpg,1,mulher,train\nreal_00498.jpg,0,mulher,train\nreal_01021.jpg,0,mulher,train\nmid_440_0101.jpg,1,mulher,train\nreal_00398.jpg,0,mulher,train\neasy_179_0001.jpg,1,mulher,train\nreal_00251.jpg,0,mulher,train\nreal_00165.jpg,0,homem,train\nreal_00296.jpg,0,mulher,train\nreal_00946.jpg,0,mulher,train\nmid_160_1110.jpg,1,mulher,train\nmid_416_1111.jpg,1,homem,train\nreal_00424.jpg,0,mulher,train\nmid_66_1100.jpg,1,mulher,train\nreal_00235.jpg,0,homem,train\nreal_01047.jpg,0,mulher,train\neasy_71_1100.jpg,1,mulher,train\nhard_194_0110.jpg,1,mulher,train\nreal_00452.jpg,0,homem,train\nreal_00812.jpg,0,mulher,train\neasy_116_111.jpg,1,homem,train\nreal_01017.jpg,0,mulher,train\neasy_52_1100.jpg,1,mulher,train\nreal_00690.jpg,0,mulher,train\nreal_00761.jpg,0,homem,train\nmid_63_1111.jpg,1,homem,train\neasy_227_1110.jpg,1,homem,train\neasy_74_0010.jpg,1,mulher,train\nreal_00349.jpg,0,mulher,train\nreal_00714.jpg,0,mulher,train\nhard_75_0011.jpg,1,mulher,train\nreal_00751.jpg,0,mulher,train\neasy_219_1101.jpg,1,homem,train\nreal_00431.jpg,0,mulher,train\neasy_169_1100.jpg,1,mulher,train\nmid_472_1101.jpg,1,homem,train\nreal_00345.jpg,0,mulher,train\nmid_418_1101.jpg,1,homem,train\nreal_00175.jpg,0,mulher,train\nreal_00809.jpg,0,mulher,train\nreal_01009.jpg,0,mulher,train\nreal_00691.jpg,0,mulher,train\neasy_17_0011.jpg,1,mulher,train\neasy_110_0001.jpg,1,mulher,train\nreal_00683.jpg,0,mulher,train\nreal_00455.jpg,0,mulher,train\nreal_00766.jpg,0,mulher,train\nreal_00116.jpg,0,homem,train\nhard_217_1100.jpg,1,homem,train\nhard_204_0011.jpg,1,homem,train\nreal_00999.jpg,0,homem,train\nmid_23_1111.jpg,1,homem,train\neasy_77_0111.jpg,1,mulher,train\nmid_250_1111.jpg,1,mulher,train\nhard_29_1111.jpg,1,mulher,train\nreal_00403.jpg,0,mulher,train\nreal_00908.jpg,0,mulher,train\nreal_00123.jpg,0,homem,train\nmid_303_1100.jpg,1,homem,train\neasy_20_1111.jpg,1,mulher,train\neasy_205_1110.jpg,1,mulher,train\nreal_00061.jpg,0,mulher,train\nreal_00935.jpg,0,homem,train\nreal_00603.jpg,0,mulher,train\nreal_01015.jpg,0,mulher,train\nreal_00073.jpg,0,mulher,train\nhard_176_0011.jpg,1,homem,train\nreal_00127.jpg,0,homem,train\nreal_00324.jpg,0,mulher,train\nhard_127_0011.jpg,1,mulher,train\nhard_17_1010.jpg,1,mulher,train\nreal_00784.jpg,0,mulher,train\nmid_106_0001.jpg,1,mulher,train\nmid_113_1100.jpg,1,mulher,train\nmid_216_1011.jpg,1,homem,train\nmid_444_1111.jpg,1,mulher,train\nmid_352_1111.jpg,1,mulher,train\nreal_00084.jpg,0,mulher,train\neasy_67_0101.jpg,1,mulher,train\nhard_208_0001.jpg,1,mulher,train\nreal_00711.jpg,0,homem,train\nmid_81_1101.jpg,1,homem,train\nreal_00114.jpg,0,homem,train\nreal_01028.jpg,0,mulher,train\nmid_3_1100.jpg,1,mulher,train\nreal_00171.jpg,0,homem,train\neasy_233_0010.jpg,1,homem,train\nreal_00896.jpg,0,homem,train\nmid_307_1101.jpg,1,homem,train\nhard_224_0111.jpg,1,mulher,train\nmid_353_0011.jpg,1,mulher,train\nmid_176_1111.jpg,1,mulher,train\nreal_00157.jpg,0,mulher,train\nreal_00184.jpg,0,homem,train\nreal_01011.jpg,0,mulher,train\nreal_00499.jpg,0,mulher,train\nreal_00056.jpg,0,homem,train\nreal_01019.jpg,0,mulher,train\nreal_00907.jpg,0,homem,train\nhard_59_1110.jpg,1,mulher,train\nmid_321_1110.jpg,1,mulher,train\nmid_132_0001.jpg,1,mulher,train\nreal_00046.jpg,0,mulher,train\nreal_00145.jpg,0,mulher,train\nreal_00777.jpg,0,mulher,train\nmid_84_1111.jpg,1,homem,train\neasy_143_0011.jpg,1,homem,train\nmid_239_1011.jpg,1,mulher,train\nmid_48_1111.jpg,1,homem,train\nreal_00552.jpg,0,mulher,train\nreal_00974.jpg,0,homem,train\neasy_125_0011.jpg,1,homem,train\nreal_00855.jpg,0,mulher,train\nreal_00377.jpg,0,homem,train\nreal_01006.jpg,0,mulher,train\nmid_117_1111.jpg,1,homem,train\nreal_00267.jpg,0,mulher,train\nmid_247_1111.jpg,1,mulher,train\nhard_225_0011.jpg,1,mulher,train\nreal_01042.jpg,0,mulher,train\neasy_99_0101.jpg,1,homem,train\nreal_00881.jpg,0,mulher,train\nreal_00457.jpg,0,mulher,train\nreal_00119.jpg,0,homem,train\nmid_172_1111.jpg,1,homem,train\nreal_00188.jpg,0,mulher,train\neasy_223_0100.jpg,1,homem,train\nreal_00987.jpg,0,mulher,train\nreal_00308.jpg,0,mulher,train\nmid_281_1111.jpg,1,mulher,train\nreal_00822.jpg,0,mulher,train\nhard_138_1101.jpg,1,homem,train\nreal_00414.jpg,0,homem,train\nhard_114_1111.jpg,1,homem,train\nmid_428_0001.jpg,1,mulher,train\nhard_207_1100.jpg,1,homem,train\nmid_305_1101.jpg,1,mulher,train\nreal_00029.jpg,0,homem,train\nhard_90_1100.jpg,1,mulher,train\nreal_00748.jpg,0,homem,train\nreal_00186.jpg,0,mulher,train\neasy_203_0111.jpg,1,mulher,train\nmid_296_1111.jpg,1,mulher,train\nreal_00365.jpg,0,mulher,train\nreal_01037.jpg,0,mulher,train\neasy_160_0011.jpg,1,homem,train\nreal_00337.jpg,0,mulher,train\nreal_00614.jpg,0,homem,train\neasy_117_0101.jpg,1,homem,train\nmid_182_1111.jpg,1,homem,train\nmid_446_0011.jpg,1,homem,train\neasy_51_1110.jpg,1,homem,train\neasy_137_0001.jpg,1,homem,train\nhard_61_0011.jpg,1,homem,train\nhard_12_1101.jpg,1,homem,train\nmid_272_1010.jpg,1,mulher,train\nreal_00057.jpg,0,homem,train\nreal_00198.jpg,0,homem,train\nreal_00584.jpg,0,mulher,train\nreal_00067.jpg,0,homem,train\nreal_01029.jpg,0,mulher,train\nmid_75_1111.jpg,1,homem,train\neasy_123_0100.jpg,1,homem,train\nreal_00208.jpg,0,homem,train\nreal_00933.jpg,0,mulher,train\neasy_63_1110.jpg,1,mulher,train\nmid_203_1111.jpg,1,mulher,train\nreal_01020.jpg,0,mulher,train\nreal_00968.jpg,0,mulher,train\neasy_210_1011.jpg,1,homem,train\nhard_71_1100.jpg,1,mulher,train\neasy_177_1010.jpg,1,mulher,train\neasy_226_1111.jpg,1,mulher,train\neasy_68_1100.jpg,1,homem,train\neasy_33_0010.jpg,1,mulher,train\nhard_72_1100.jpg,1,homem,train\nreal_00588.jpg,0,mulher,train\nhard_183_1101.jpg,1,homem,val\nhard_227_1100.jpg,1,homem,val\nreal_00897.jpg,0,homem,val\nhard_53_1100.jpg,1,mulher,val\nmid_385_1100.jpg,1,mulher,val\nreal_00142.jpg,0,homem,val\nhard_151_0101.jpg,1,homem,val\nhard_60_1111.jpg,1,homem,val\nreal_00873.jpg,0,mulher,val\nmid_426_1111.jpg,1,mulher,val\nreal_00994.jpg,0,homem,val\nmid_336_1111.jpg,1,homem,val\nreal_00734.jpg,0,homem,val\nmid_41_1111.jpg,1,mulher,val\nhard_40_1111.jpg,1,homem,val\nhard_120_0011.jpg,1,mulher,val\nmid_355_1011.jpg,1,mulher,val\nreal_00633.jpg,0,homem,val\nmid_16_1111.jpg,1,mulher,val\nreal_00773.jpg,0,mulher,val\nreal_00092.jpg,0,homem,val\nhard_39_1111.jpg,1,homem,val\nreal_00362.jpg,0,mulher,val\nreal_01032.jpg,0,mulher,val\neasy_232_1100.jpg,1,mulher,val\neasy_162_0001.jpg,1,mulher,val\nreal_00682.jpg,0,homem,val\nreal_00786.jpg,0,mulher,val\nreal_00586.jpg,0,homem,val\neasy_140_1111.jpg,1,mulher,val\nreal_00063.jpg,0,mulher,val\nreal_01002.jpg,0,mulher,val\nreal_00283.jpg,0,mulher,val\nreal_00471.jpg,0,mulher,val\nreal_00190.jpg,0,mulher,val\nmid_20_1111.jpg,1,mulher,val\nreal_00509.jpg,0,mulher,val\neasy_119_0011.jpg,1,mulher,val\nhard_159_0100.jpg,1,mulher,val\nreal_00330.jpg,0,mulher,val\nreal_00086.jpg,0,homem,val\nreal_00765.jpg,0,mulher,val\nhard_122_1100.jpg,1,mulher,val\nreal_00926.jpg,0,homem,val\neasy_75_1100.jpg,1,mulher,val\nreal_00607.jpg,0,mulher,val\nreal_00024.jpg,0,mulher,val\nreal_00239.jpg,0,homem,val\nreal_00753.jpg,0,homem,val\nreal_00995.jpg,0,mulher,val\nreal_00246.jpg,0,mulher,val\nmid_309_1111.jpg,1,mulher,val\nhard_228_1111.jpg,1,homem,val\neasy_46_1100.jpg,1,mulher,val\nmid_295_0111.jpg,1,mulher,val\neasy_3_1100.jpg,1,mulher,val\nreal_00102.jpg,0,mulher,val\nreal_00525.jpg,0,homem,val\nreal_00783.jpg,0,mulher,val\nmid_435_1111.jpg,1,homem,val\nreal_00736.jpg,0,mulher,val\nmid_47_1110.jpg,1,homem,val\neasy_190_0101.jpg,1,mulher,val\nreal_00465.jpg,0,homem,val\nreal_00785.jpg,0,mulher,val\nhard_10_1111.jpg,1,homem,val\nreal_01060.jpg,0,homem,val\nhard_47_1100.jpg,1,mulher,val\neasy_58_0011.jpg,1,homem,val\nmid_373_1111.jpg,1,homem,val\nmid_414_0011.jpg,1,homem,val\nreal_00052.jpg,0,mulher,val\nreal_00050.jpg,0,homem,val\nreal_00662.jpg,0,homem,val\nreal_00273.jpg,0,mulher,val\nmid_32_1111.jpg,1,mulher,val\nhard_34_1111.jpg,1,mulher,val\nhard_164_1000.jpg,1,mulher,val\nhard_160_1111.jpg,1,mulher,val\neasy_192_1000.jpg,1,mulher,val\nreal_00451.jpg,0,mulher,val\nreal_00877.jpg,0,homem,val\nmid_313_1101.jpg,1,mulher,val\nreal_00941.jpg,0,homem,val\nreal_00737.jpg,0,mulher,val\nreal_00204.jpg,0,mulher,val\nreal_00523.jpg,0,homem,val\nmid_288_1101.jpg,1,mulher,val\nmid_439_1000.jpg,1,homem,val\nreal_00415.jpg,0,homem,val\nreal_00600.jpg,0,mulher,val\nhard_73_0011.jpg,1,homem,val\nreal_00678.jpg,0,mulher,val\nhard_32_1111.jpg,1,homem,val\nreal_00112.jpg,0,mulher,val\nreal_00250.jpg,0,mulher,val\nreal_00180.jpg,0,homem,val\nreal_00565.jpg,0,homem,val\neasy_27_1000.jpg,1,mulher,val\nmid_406_0011.jpg,1,mulher,val\nreal_00825.jpg,0,mulher,val\nhard_65_0101.jpg,1,mulher,val\nmid_107_0010.jpg,1,mulher,val\nmid_130_1111.jpg,1,mulher,val\nmid_171_1100.jpg,1,mulher,val\nmid_301_0101.jpg,1,mulher,val\nmid_139_1111.jpg,1,homem,val\nmid_398_0011.jpg,1,homem,val\neasy_222_0011.jpg,1,mulher,val\nreal_00083.jpg,0,homem,val\nreal_00081.jpg,0,mulher,val\nreal_00335.jpg,0,mulher,val\nmid_348_1100.jpg,1,mulher,val\nhard_213_0101.jpg,1,homem,val\nhard_6_1110.jpg,1,homem,val\nreal_00768.jpg,0,mulher,val\nreal_00194.jpg,0,homem,val\nreal_00978.jpg,0,mulher,val\nreal_00280.jpg,0,mulher,val\nmid_180_1110.jpg,1,mulher,val\neasy_88_0011.jpg,1,mulher,val\nreal_00004.jpg,0,homem,val\nhard_142_1110.jpg,1,homem,val\nhard_201_0100.jpg,1,homem,val\nmid_186_1100.jpg,1,mulher,val\neasy_158_0011.jpg,1,mulher,val\nmid_56_1111.jpg,1,mulher,val\nreal_00215.jpg,0,mulher,val\nmid_442_0111.jpg,1,homem,val\nreal_00395.jpg,0,mulher,val\nmid_175_1111.jpg,1,mulher,val\nreal_00663.jpg,0,mulher,val\nreal_00724.jpg,0,homem,val\nmid_285_1111.jpg,1,mulher,val\nreal_00307.jpg,0,mulher,val\neasy_84_0011.jpg,1,mulher,val\neasy_15_0011.jpg,1,homem,val\nhard_55_0011.jpg,1,mulher,val\nhard_174_1111.jpg,1,homem,val\nreal_00962.jpg,0,homem,val\nreal_00816.jpg,0,mulher,val\nreal_00375.jpg,0,mulher,val\nreal_00589.jpg,0,homem,val\nreal_00210.jpg,0,mulher,val\neasy_213_1111.jpg,1,homem,val\nmid_114_1110.jpg,1,homem,val\nreal_00272.jpg,0,mulher,val\nmid_366_1111.jpg,1,homem,val\nreal_00193.jpg,0,homem,val\nhard_211_0110.jpg,1,mulher,val\nhard_233_0010.jpg,1,mulher,val\nmid_453_1111.jpg,1,mulher,val\nreal_00885.jpg,0,homem,val\nreal_00917.jpg,0,mulher,val\nhard_189_0011.jpg,1,homem,val\nmid_450_1101.jpg,1,homem,val\nreal_00240.jpg,0,mulher,val\nhard_36_1101.jpg,1,mulher,val\nreal_00771.jpg,0,mulher,val\nmid_40_1111.jpg,1,homem,val\neasy_118_1111.jpg,1,mulher,val\nreal_00017.jpg,0,homem,val\nreal_00834.jpg,0,homem,val\nreal_00299.jpg,0,mulher,val\nreal_00749.jpg,0,homem,val\nreal_00889.jpg,0,homem,val\neasy_66_1101.jpg,1,mulher,val\nreal_00288.jpg,0,mulher,val\nreal_00359.jpg,0,mulher,val\nmid_163_1001.jpg,1,homem,val\nmid_39_1111.jpg,1,mulher,val\nreal_00818.jpg,0,mulher,val\neasy_171_1100.jpg,1,mulher,val\nreal_01072.jpg,0,homem,val\nreal_00407.jpg,0,homem,val\neasy_161_1101.jpg,1,mulher,val\nhard_158_0011.jpg,1,mulher,val\nreal_00181.jpg,0,mulher,val\nreal_00832.jpg,0,mulher,val\nreal_00120.jpg,0,homem,val\nreal_00719.jpg,0,homem,val\neasy_108_1111.jpg,1,mulher,val\nreal_01023.jpg,0,mulher,val\nreal_01075.jpg,0,mulher,val\nreal_00014.jpg,0,mulher,val\nreal_00306.jpg,0,mulher,val\nmid_190_1111.jpg,1,mulher,val\nmid_280_1111.jpg,1,homem,val\nreal_00951.jpg,0,homem,val\nreal_00111.jpg,0,homem,val\nreal_01035.jpg,0,mulher,val\nreal_00993.jpg,0,homem,val\nreal_00615.jpg,0,homem,val\nreal_00018.jpg,0,mulher,val\nreal_00643.jpg,0,homem,val\nmid_185_1110.jpg,1,homem,val\nreal_00596.jpg,0,mulher,val\nreal_00408.jpg,0,homem,val\neasy_103_1111.jpg,1,mulher,val\nreal_00841.jpg,0,homem,val\nreal_00542.jpg,0,mulher,val\nreal_00581.jpg,0,homem,val\nreal_00526.jpg,0,mulher,val\nreal_00030.jpg,0,homem,val\nreal_00108.jpg,0,homem,val\nreal_00616.jpg,0,homem,val\nmid_329_1110.jpg,1,homem,val\nreal_01076.jpg,0,mulher,val\nhard_22_1111.jpg,1,mulher,val\nmid_456_1111.jpg,1,homem,val\nreal_00468.jpg,0,mulher,val\nreal_00287.jpg,0,mulher,val\nhard_1_1111.jpg,1,mulher,val\nreal_00689.jpg,0,homem,val\nreal_00508.jpg,0,homem,val\nmid_127_0011.jpg,1,mulher,val\nreal_00973.jpg,0,mulher,val\neasy_133_1100.jpg,1,homem,val\neasy_187_1110.jpg,1,homem,val\nreal_00197.jpg,0,mulher,val\nmid_61_1111.jpg,1,homem,val\nmid_197_1110.jpg,1,mulher,val\nmid_43_1111.jpg,1,mulher,val\nreal_00963.jpg,0,mulher,val\neasy_101_0010.jpg,1,homem,val\nreal_00721.jpg,0,homem,val\nreal_00660.jpg,0,mulher,val\nhard_100_1111.jpg,1,mulher,val\nreal_01080.jpg,0,homem,val\nmid_60_1101.jpg,1,homem,val\nreal_00658.jpg,0,homem,val\nmid_361_0001.jpg,1,homem,val\nhard_97_1001.jpg,1,mulher,val\nreal_00649.jpg,0,mulher,val\nhard_203_0010.jpg,1,mulher,val\nreal_00985.jpg,0,mulher,val\nreal_00984.jpg,0,mulher,val\nreal_00560.jpg,0,homem,val\nmid_194_1111.jpg,1,mulher,val\nreal_00911.jpg,0,mulher,val\nreal_00731.jpg,0,mulher,val\nmid_242_1111.jpg,1,homem,val\nreal_00860.jpg,0,mulher,val\nmid_458_1111.jpg,1,homem,val\nreal_00166.jpg,0,homem,val\nmid_330_1111.jpg,1,mulher,val\nreal_00801.jpg,0,mulher,val\nreal_00421.jpg,0,homem,val\nhard_106_1100.jpg,1,mulher,val\nmid_62_1110.jpg,1,homem,val\nhard_4_1111.jpg,1,homem,val\nreal_00284.jpg,0,mulher,val\nreal_00692.jpg,0,homem,val\nreal_01022.jpg,0,mulher,val\nreal_00573.jpg,0,homem,val\nmid_360_1100.jpg,1,mulher,val\neasy_144_1111.jpg,1,mulher,val\nreal_00265.jpg,0,mulher,val\nreal_01066.jpg,0,homem,val\neasy_5_1100.jpg,1,mulher,val\nreal_00078.jpg,0,homem,val\nreal_00464.jpg,0,homem,val\nreal_01030.jpg,0,mulher,val\nmid_322_1111.jpg,1,mulher,val\neasy_122_1011.jpg,1,homem,val\nmid_178_1111.jpg,1,mulher,val\nmid_91_1111.jpg,1,mulher,val\nmid_269_1100.jpg,1,homem,val\nmid_275_1100.jpg,1,homem,val\nreal_00840.jpg,0,mulher,val\nreal_00480.jpg,0,mulher,val\nhard_177_1011.jpg,1,mulher,val\nreal_00609.jpg,0,mulher,val\nmid_198_1111.jpg,1,mulher,val\nreal_00810.jpg,0,homem,val\nmid_169_0001.jpg,1,mulher,val\nreal_00085.jpg,0,mulher,val\nreal_00302.jpg,0,mulher,val\nmid_432_1111.jpg,1,homem,val\nhard_139_1011.jpg,1,homem,val\nreal_00009.jpg,0,homem,val\nreal_00778.jpg,0,mulher,val\nreal_00064.jpg,0,homem,val\nhard_25_1111.jpg,1,mulher,val\nhard_129_1111.jpg,1,homem,val\neasy_50_0110.jpg,1,mulher,val\nreal_00795.jpg,0,mulher,val\nhard_13_1111.jpg,1,homem,val\nreal_00880.jpg,0,mulher,val\nreal_00906.jpg,0,homem,val\nreal_00185.jpg,0,homem,val\nmid_87_1111.jpg,1,homem,val\nreal_00357.jpg,0,mulher,val\nhard_220_1111.jpg,1,homem,val\nmid_400_0011.jpg,1,homem,val\neasy_4_0011.jpg,1,mulher,val\neasy_212_0010.jpg,1,mulher,val\nreal_00675.jpg,0,mulher,val\nmid_278_1101.jpg,1,mulher,val\nreal_00309.jpg,0,mulher,val\nhard_113_0011.jpg,1,mulher,val\nmid_356_1110.jpg,1,homem,val\nreal_00696.jpg,0,mulher,val\nreal_00172.jpg,0,mulher,val\nmid_402_0011.jpg,1,homem,val\nreal_00939.jpg,0,homem,val\nreal_00275.jpg,0,mulher,test\nmid_431_0001.jpg,1,mulher,test\nmid_335_1100.jpg,1,homem,test\nreal_00436.jpg,0,mulher,test\nreal_00651.jpg,0,mulher,test\nreal_00725.jpg,0,mulher,test\nmid_347_0011.jpg,1,mulher,test\nreal_00094.jpg,0,homem,test\nhard_93_1111.jpg,1,mulher,test\nmid_241_1111.jpg,1,mulher,test\nreal_00400.jpg,0,homem,test\nreal_00571.jpg,0,homem,test\nreal_00304.jpg,0,mulher,test\nreal_00534.jpg,0,homem,test\nreal_00707.jpg,0,mulher,test\nreal_00293.jpg,0,mulher,test\neasy_208_1111.jpg,1,mulher,test\nhard_146_0110.jpg,1,homem,test\nmid_33_1011.jpg,1,mulher,test\neasy_61_0111.jpg,1,homem,test\nreal_00443.jpg,0,mulher,test\nreal_00459.jpg,0,homem,test\nmid_99_1100.jpg,1,mulher,test\nreal_00179.jpg,0,homem,test\nreal_01004.jpg,0,homem,test\neasy_234_1000.jpg,1,mulher,test\nmid_17_1111.jpg,1,homem,test\nmid_53_1100.jpg,1,homem,test\nreal_00390.jpg,0,homem,test\neasy_172_1001.jpg,1,mulher,test\nreal_00259.jpg,0,homem,test\nreal_00331.jpg,0,mulher,test\nhard_156_1111.jpg,1,homem,test\nreal_00136.jpg,0,mulher,test\nmid_158_1010.jpg,1,mulher,test\nreal_00938.jpg,0,mulher,test\nreal_00756.jpg,0,homem,test\nreal_00535.jpg,0,homem,test\nreal_00548.jpg,0,mulher,test\nreal_00760.jpg,0,mulher,test\nmid_159_1111.jpg,1,mulher,test\nmid_345_1111.jpg,1,mulher,test\nreal_00434.jpg,0,mulher,test\nreal_00992.jpg,0,mulher,test\nreal_00598.jpg,0,homem,test\nreal_00316.jpg,0,mulher,test\nreal_00093.jpg,0,homem,test\nreal_00271.jpg,0,mulher,test\nreal_00342.jpg,0,mulher,test\nmid_395_1110.jpg,1,mulher,test\nmid_391_1110.jpg,1,homem,test\nmid_133_1001.jpg,1,mulher,test\nhard_102_1001.jpg,1,mulher,test\nreal_00975.jpg,0,mulher,test\neasy_164_1111.jpg,1,homem,test\nmid_59_1111.jpg,1,mulher,test\nreal_00988.jpg,0,homem,test\nreal_00474.jpg,0,mulher,test\nreal_00106.jpg,0,mulher,test\nreal_00767.jpg,0,mulher,test\nreal_00654.jpg,0,homem,test\nreal_00604.jpg,0,homem,test\nmid_339_1100.jpg,1,mulher,test\nmid_134_1010.jpg,1,mulher,test\nmid_267_1111.jpg,1,mulher,test\nreal_00722.jpg,0,homem,test\nreal_00594.jpg,0,homem,test\nreal_00741.jpg,0,homem,test\neasy_132_0110.jpg,1,mulher,test\nreal_00582.jpg,0,homem,test\nmid_29_1111.jpg,1,homem,test\neasy_182_0100.jpg,1,mulher,test\nreal_00639.jpg,0,mulher,test\nmid_332_1111.jpg,1,homem,test\nreal_00485.jpg,0,mulher,test\nreal_01007.jpg,0,homem,test\nmid_369_1110.jpg,1,homem,test\nmid_164_0011.jpg,1,homem,test\neasy_142_1111.jpg,1,homem,test\nhard_18_1111.jpg,1,mulher,test\neasy_47_1110.jpg,1,homem,test\nmid_248_0111.jpg,1,mulher,test\nreal_00720.jpg,0,mulher,test\nmid_251_1111.jpg,1,mulher,test\nreal_00162.jpg,0,homem,test\nreal_00478.jpg,0,mulher,test\nhard_152_0111.jpg,1,mulher,test\nreal_00754.jpg,0,mulher,test\nreal_00990.jpg,0,homem,test\nreal_00437.jpg,0,mulher,test\nreal_00500.jpg,0,homem,test\nhard_170_0111.jpg,1,mulher,test\nreal_00356.jpg,0,mulher,test\nreal_00264.jpg,0,homem,test\nmid_122_1100.jpg,1,homem,test\neasy_151_1110.jpg,1,mulher,test\nmid_244_1110.jpg,1,mulher,test\neasy_91_1000.jpg,1,mulher,test\nmid_145_1111.jpg,1,mulher,test\nreal_00031.jpg,0,homem,test\neasy_31_0011.jpg,1,homem,test\nmid_465_1111.jpg,1,homem,test\nreal_00702.jpg,0,homem,test\nreal_00099.jpg,0,homem,test\nmid_279_1111.jpg,1,mulher,test\nmid_151_1100.jpg,1,mulher,test\nreal_00054.jpg,0,mulher,test\nreal_00257.jpg,0,mulher,test\nmid_220_0011.jpg,1,mulher,test\nreal_00666.jpg,0,homem,test\nreal_01046.jpg,0,mulher,test\nreal_00329.jpg,0,mulher,test\nmid_65_1111.jpg,1,mulher,test\nreal_00872.jpg,0,mulher,test\nreal_00456.jpg,0,mulher,test\nhard_20_1111.jpg,1,homem,test\nmid_141_1110.jpg,1,mulher,test\neasy_136_0111.jpg,1,homem,test\nmid_68_1111.jpg,1,homem,test\nhard_123_1111.jpg,1,mulher,test\nmid_147_1110.jpg,1,mulher,test\neasy_85_1111.jpg,1,homem,test\nreal_00313.jpg,0,mulher,test\nmid_344_0010.jpg,1,homem,test\nmid_299_1100.jpg,1,homem,test\nmid_389_1101.jpg,1,mulher,test\nreal_01031.jpg,0,mulher,test\nreal_00506.jpg,0,homem,test\nreal_00882.jpg,0,mulher,test\nreal_00566.jpg,0,homem,test\nmid_477_1100.jpg,1,mulher,test\nreal_00164.jpg,0,mulher,test\nhard_77_1100.jpg,1,homem,test\nmid_202_1110.jpg,1,homem,test\nmid_423_1101.jpg,1,homem,test\nreal_00617.jpg,0,mulher,test\nmid_364_1111.jpg,1,homem,test\nmid_306_1111.jpg,1,mulher,test\nreal_00087.jpg,0,homem,test\nreal_00254.jpg,0,homem,test\nhard_145_1100.jpg,1,mulher,test\nmid_149_0011.jpg,1,mulher,test\nreal_00010.jpg,0,mulher,test\nmid_140_1111.jpg,1,mulher,test\neasy_180_0011.jpg,1,mulher,test\nmid_443_0011.jpg,1,mulher,test\nreal_00954.jpg,0,homem,test\nmid_376_0001.jpg,1,homem,test\nreal_00634.jpg,0,mulher,test\nreal_00813.jpg,0,mulher,test\nmid_461_1100.jpg,1,homem,test\nreal_00343.jpg,0,mulher,test\nmid_14_1101.jpg,1,mulher,test\nreal_00167.jpg,0,mulher,test\nhard_143_0100.jpg,1,homem,test\nmid_419_0011.jpg,1,mulher,test\nmid_50_1111.jpg,1,homem,test\nreal_00290.jpg,0,mulher,test\nmid_392_1111.jpg,1,homem,test\nhard_78_1101.jpg,1,homem,test\nmid_179_1111.jpg,1,mulher,test\nreal_00412.jpg,0,homem,test\nreal_00843.jpg,0,mulher,test\nhard_195_1000.jpg,1,homem,test\nhard_134_1100.jpg,1,homem,test\nhard_111_1111.jpg,1,homem,test\neasy_173_0111.jpg,1,mulher,test\neasy_224_1100.jpg,1,mulher,test\nmid_240_1100.jpg,1,homem,test\nreal_00811.jpg,0,mulher,test\nreal_00060.jpg,0,homem,test\nreal_00839.jpg,0,mulher,test\nreal_00002.jpg,0,mulher,test\nreal_00037.jpg,0,homem,test\nmid_173_1110.jpg,1,homem,test\nreal_01026.jpg,0,mulher,test\nreal_00072.jpg,0,homem,test\nmid_10_1111.jpg,1,mulher,test\nhard_239_1000.jpg,1,homem,test\nmid_292_1000.jpg,1,homem,test\nhard_86_1100.jpg,1,homem,test\neasy_134_0001.jpg,1,homem,test\nmid_434_1110.jpg,1,homem,test\nreal_01001.jpg,0,homem,test\nreal_00931.jpg,0,mulher,test\nreal_00879.jpg,0,mulher,test\neasy_92_0011.jpg,1,mulher,test\nmid_362_1101.jpg,1,mulher,test\nreal_00105.jpg,0,homem,test\nreal_00533.jpg,0,mulher,test\nhard_186_0011.jpg,1,homem,test\nreal_00858.jpg,0,mulher,test\nmid_337_1111.jpg,1,homem,test\nreal_00815.jpg,0,homem,test\neasy_148_0011.jpg,1,homem,test\nreal_00154.jpg,0,mulher,test\nreal_00237.jpg,0,homem,test\nmid_433_0011.jpg,1,homem,test\nhard_231_1111.jpg,1,mulher,test\nreal_00352.jpg,0,mulher,test\nhard_63_1101.jpg,1,mulher,test\nreal_00556.jpg,0,homem,test\nreal_00817.jpg,0,mulher,test\nreal_00321.jpg,0,mulher,test\nmid_422_0011.jpg,1,mulher,test\nreal_00028.jpg,0,mulher,test\neasy_188_1111.jpg,1,mulher,test\nreal_00319.jpg,0,mulher,test\nreal_00044.jpg,0,mulher,test\nreal_00706.jpg,0,homem,test\nhard_28_1111.jpg,1,homem,test\nmid_231_1100.jpg,1,mulher,test\nreal_00492.jpg,0,homem,test\nhard_235_0001.jpg,1,homem,test\nreal_00769.jpg,0,homem,test\nreal_00903.jpg,0,mulher,test\nreal_00530.jpg,0,homem,test\nreal_00510.jpg,0,mulher,test\nreal_00991.jpg,0,mulher,test\nmid_11_1100.jpg,1,mulher,test\nreal_00952.jpg,0,homem,test\nhard_48_0011.jpg,1,mulher,test\nreal_00623.jpg,0,homem,test\nmid_327_0111.jpg,1,mulher,test\nmid_273_1101.jpg,1,mulher,test\nreal_00392.jpg,0,homem,test\nhard_108_1111.jpg,1,mulher,test\nreal_00829.jpg,0,mulher,test\nreal_00936.jpg,0,mulher,test\neasy_115_0010.jpg,1,mulher,test\nmid_470_1001.jpg,1,mulher,test\nreal_01038.jpg,0,mulher,test\nreal_00774.jpg,0,mulher,test\nreal_00325.jpg,0,mulher,test\nmid_184_1101.jpg,1,mulher,test\nhard_99_1111.jpg,1,mulher,test\nhard_11_1100.jpg,1,homem,test\nhard_214_1011.jpg,1,mulher,test\nreal_00462.jpg,0,homem,test\nreal_00227.jpg,0,homem,test\nhard_24_1100.jpg,1,mulher,test\nmid_380_1111.jpg,1,mulher,test\nreal_00354.jpg,0,mulher,test\nhard_21_1101.jpg,1,mulher,test\nmid_393_1101.jpg,1,mulher,test\nreal_00341.jpg,0,mulher,test\nmid_70_1111.jpg,1,mulher,test\nreal_00610.jpg,0,homem,test\nreal_00851.jpg,0,homem,test\nreal_00089.jpg,0,mulher,test\nreal_00493.jpg,0,mulher,test\nreal_00159.jpg,0,homem,test\nreal_00090.jpg,0,homem,test\nreal_00216.jpg,0,mulher,test\nreal_00101.jpg,0,mulher,test\nhard_131_1000.jpg,1,homem,test\nreal_00674.jpg,0,homem,test\nmid_371_0011.jpg,1,homem,test\nreal_00697.jpg,0,mulher,test\nreal_00418.jpg,0,homem,test\nmid_377_0100.jpg,1,homem,test\nreal_00688.jpg,0,mulher,test\nhard_80_0100.jpg,1,mulher,test\nreal_00373.jpg,0,mulher,test\nreal_00590.jpg,0,homem,test\nreal_00463.jpg,0,mulher,test\nreal_00551.jpg,0,mulher,test\nreal_00453.jpg,0,homem,test\nreal_00895.jpg,0,homem,test\nreal_00620.jpg,0,mulher,test\nreal_00333.jpg,0,mulher,test\nreal_00762.jpg,0,homem,test\nmid_108_1110.jpg,1,mulher,test\nhard_175_1100.jpg,1,homem,test\neasy_129_1111.jpg,1,mulher,test\nmid_378_1110.jpg,1,mulher,test\nreal_00888.jpg,0,mulher,test\neasy_216_1100.jpg,1,homem,test\nreal_00178.jpg,0,homem,test\neasy_221_0111.jpg,1,homem,test\nmid_196_1100.jpg,1,homem,test\nreal_00406.jpg,0,homem,test\nreal_00312.jpg,0,mulher,test\nreal_00512.jpg,0,mulher,test\nreal_00981.jpg,0,mulher,test\nhard_218_0100.jpg,1,mulher,test\nreal_00212.jpg,0,mulher,test\nmid_136_0001.jpg,1,homem,test\nreal_00698.jpg,0,homem,test\neasy_18_0011.jpg,1,mulher,test\nreal_00038.jpg,0,homem,test\nmid_78_1101.jpg,1,mulher,test\nreal_00532.jpg,0,homem,test\nreal_00686.jpg,0,homem,test\nhard_87_1111.jpg,1,mulher,test\neasy_236_1100.jpg,1,mulher,test\nreal_00982.jpg,0,mulher,test\nreal_00143.jpg,0,mulher,test\nreal_00837.jpg,0,mulher,test\nreal_00605.jpg,0,homem,test\nreal_00369.jpg,0,mulher,test\nreal_01073.jpg,0,mulher,test\nmid_129_0101.jpg,1,homem,test\neasy_166_1100.jpg,1,mulher,test\nreal_00155.jpg,0,mulher,test\nmid_427_1010.jpg,1,mulher,test\neasy_90_1111.jpg,1,mulher,test\n'))


In [ ]:
assert clean.filename.is_unique and adv.filename.is_unique and split.filename.is_unique
joined=clean.merge(split[['filename','split']],on='filename',how='left',validate='one_to_one')
assert joined.split.eq('test').all()
pair=clean.merge(adv,on=['filename','label','gender'],validate='one_to_one',suffixes=('_clean','_adv'))
assert len(pair)==len(clean)==len(adv)
assert (pair.pred==pair.pred_limpa).all()
assert (clean.acerto==(clean.pred==clean.label).astype(int)).all()
assert (adv.ataque_bem_sucedido==((adv.pred_limpa==adv.label)&(adv.pred_adversarial!=adv.label)).astype(int)).all()
rows=[]
for group,d in pair.groupby('gender'):
    r=summarize(d);eligible=d[d.pred_limpa==d.label]
    k=int((eligible.pred_adversarial!=eligible.label).sum());n=len(eligible)
    r.update(group=group,asr=k/n,n_asr=n,success=k)
    r['accuracy_low'],r['accuracy_high']=wilson(int((d.pred==d.label).sum()),len(d))
    r['asr_low'],r['asr_high']=wilson(k,n)
    rows.append(r)
summary=pd.DataFrame(rows)
display(summary)
print('Acurácia global:',(clean.pred==clean.label).mean())
from scipy.stats import fisher_exact
for name,a,b in [('FPR',[[13,56],[35,59]],None),('FNR',[[27,32],[30,55]],None)]:
    print(name,'Fisher exploratório (não ajustado):',fisher_exact(a).pvalue)
summary.to_csv('resultados_historicos_recalculados.csv',index=False)


## 2. Como interpretar
A diferença de acurácia não resume os erros: o grupo “mulher” anotado tem FPR maior, enquanto “homem” tem FNR maior. ASRs próximas e elevadas mostram vulnerabilidade em ambos. As comparações são exploratórias; IDs de pessoa e independência das imagens não foram estabelecidos. Não concluir causalidade.

O código antigo aplica epsilon 8/255 em tensores normalizados em [-1,1]; equivale a 4/255 em [0,1]. Este notebook recalcula os CSVs existentes, não valida a execução do treinamento, checkpoint, qualidade dos rótulos ou geração original.
